# Teder: AI Music Genre Classifier

Teder uses MFCC (Mel-frequency cepstral coefficient) features and a convolutional neural network to classify an audio track into one of ten music genres.

The application later combines the predicted genre probabilities with each user's saved genre preferences to produce a personalized Like or Dislike recommendation.

The notebook can train the CNN with augmentation, evaluate it on validation and test splits, save metrics and model metadata, and expose model-management tools through Gradio.

##Connecting to drive

In [ ]:
import os
import sys

websiteAutorun = os.environ.get('BEATSENSE_AUTORUN_MAIN_TRAINING') == '1'
inColab = 'google.colab' in sys.modules

if inColab and not websiteAutorun:
    from google.colab import drive
    drive.mount('/content/drive')
elif websiteAutorun:
    print('Skipping Google Drive mount during website-triggered training.')
else:
    print('Google Drive mount skipped outside Colab.')


Mounted at /content/drive


##libraries install and imports

In [ ]:
import os
import subprocess
import sys

if os.environ.get('BEATSENSE_AUTORUN_MAIN_TRAINING') == '1':
    print('Skipping yt-dlp installation during website-triggered training.')
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'yt-dlp'])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import tensorflow as tf
import gradio as gr
try:
    import yt_dlp
except ImportError:
    yt_dlp = None
import pytz

try:
    from IPython.display import clear_output
    clear_output()
except:
    pass

from sklearn.utils import class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout,
    GlobalAveragePooling2D, BatchNormalization, Activation, Input,
    GaussianNoise, LeakyReLU, Flatten
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, Callback, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import BinaryCrossentropy


##Constants and Configuration


In [ ]:
import os

# ---------------------------------------------------------
# CONSTANTS - No magic numbers
# ---------------------------------------------------------
# MFCC Configuration
os.environ["OMP_NUM_THREADS"] = "1"
IMG_HEIGHT = 40
IMG_WIDTH = 200
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, 1)
N_MFCC = 40
HOP_LENGTH = 512

# Audio Configuration
SAMPLE_RATE = 22050
DURATION_SECONDS = 4.0
TARGET_SAMPLES = int(SAMPLE_RATE * DURATION_SECONDS)

# Augmentation Configuration
NOISE_FACTOR = 0.0035
TIME_STRETCH_MIN = 0.92
TIME_STRETCH_MAX = 1.08
ENABLE_TIME_STRETCH_AUGMENTATION = True
COMPOSITE_CROSSFADE_SAMPLES = 2048
ENABLE_COMPOSITE_GENRE_AUGMENTATION = True
COMPOSITE_AUGMENTATION_LIMIT = 96
COMPOSITE_VARIANTS = 2
# Every original file is loaded. Expensive augmentation is limited to keep
# the doubled dataset within a practical Colab memory and runtime budget.
AUGMENTATION_FILE_LIMIT = 96

# Training Configuration
EPOCHS = 60
BATCH_SIZE = 24
LEARNING_RATE = 0.0005
L2_REGULARIZATION = 0.0032
LABEL_SMOOTHING = 0.05
EARLY_STOPPING_PATIENCE = 6
LR_REDUCTION_PATIENCE = 4
LR_REDUCTION_FACTOR = 0.4
MAX_FILES_PER_LABEL = 260
SEED = 42
PROGRESS_LOG_EVERY = 8
AUDIO_EXTENSIONS = ('.wav', '.mp3')
EXCLUDED_DATA_FOLDERS = {'like', 'dislike'}
DATA_SPLITS = ('train', 'val', 'test')


def resolve_base_project_path():
    env_path = os.environ.get("BEATSENSE_BASE_PATH", "").strip()
    if env_path:
        return env_path

    cwd = os.getcwd()
    cwd_markers = [
        os.path.join(cwd, "main.ipynb"),
        os.path.join(cwd, "app.py"),
        os.path.join(cwd, "templates"),
    ]
    if any(os.path.exists(marker) for marker in cwd_markers):
        return cwd

    if os.path.exists("/content/drive/MyDrive/hila ben levy/final-project:BeatSense"):
        return "/content/drive/MyDrive/hila ben levy/final-project:BeatSense"

    return os.path.join(cwd, "BeatSense_Project")


# File paths - AUTO DETECT ENVIRONMENT
BASE_PROJECT_PATH = resolve_base_project_path()
os.makedirs(BASE_PROJECT_PATH, exist_ok=True)
DATASET_PATH = os.path.join(BASE_PROJECT_PATH, "data/processed_data")
BEST_MODEL_PATH = os.path.join(BASE_PROJECT_PATH, "best_model.keras")
SCALER_PATH = os.path.join(BASE_PROJECT_PATH, "mfcc_scaler.pkl")
HISTORY_DIR = os.path.join(BASE_PROJECT_PATH, "models_history")
METRICS_DIR = os.path.join(BASE_PROJECT_PATH, "metrics")
os.makedirs(HISTORY_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

# Prediction Configuration
MULTI_SEGMENT_TIMESTAMPS = [0.2, 0.4, 0.5, 0.6, 0.8]
SEGMENT_WEIGHT_AVG = 0.7
SEGMENT_WEIGHT_MAX = 0.3

# File size limits (bytes)
MAX_FILE_SIZE_MB = 50
MAX_FILE_SIZE_BYTES = MAX_FILE_SIZE_MB * 1024 * 1024

# Global state
global_model = None
global_scaler = None
STOP_TRAINING_FLAG = False



## Validation Helper Functions

code cell containing utility functions for validating audio files and YouTube URLs.


In [ ]:
import os

# ---------------------------------------------------------
# VALIDATION HELPERS
# ---------------------------------------------------------
def validate_audio_file(file_path):
    """
    Validate audio file exists and is within size limits
    Input: file_path (str)
    Output: tuple (is_valid: bool, error_message: str)
    """
    if not file_path or not os.path.exists(file_path):
        return False, "File does not exist"

    file_size = os.path.getsize(file_path)
    if file_size > MAX_FILE_SIZE_BYTES:
        return False, f"File too large. Max size: {MAX_FILE_SIZE_MB}MB"

    if file_size == 0:
        return False, "File is empty"

    return True, ""

def validate_url(url):
    """
    Validate YouTube URL format
    Input: url (str)
    Output: bool
    """
    if not url:
        return False

    valid_domains = ['youtube.com', 'youtu.be', 'music.youtube.com']
    return any(domain in url.lower() for domain in valid_domains)


## extract the data preprocessing functions




In [ ]:
import librosa
import numpy as np
import pickle

# ---------------------------------------------------------
# DATA PREPROCESSING - MFCC with shared feature scaling
# ---------------------------------------------------------
def apply_feature_scaler(mfcc_flat, scaler=None):
    if scaler is None:
        mean_value = np.mean(mfcc_flat, axis=1, keepdims=True)
        std_value = np.std(mfcc_flat, axis=1, keepdims=True) + 1e-6
        return (mfcc_flat - mean_value) / std_value

    if isinstance(scaler, dict) and scaler.get('kind') == 'tensor_stats':
        mean_value = np.asarray(scaler.get('mean', []), dtype=np.float32).reshape(1, -1)
        std_value = np.asarray(scaler.get('std', []), dtype=np.float32).reshape(1, -1)
        if mean_value.size == mfcc_flat.shape[1] and std_value.size == mfcc_flat.shape[1]:
            return (mfcc_flat - mean_value) / (std_value + 1e-6)

    if hasattr(scaler, 'transform'):
        return scaler.transform(mfcc_flat)

    mean_value = np.mean(mfcc_flat, axis=1, keepdims=True)
    std_value = np.std(mfcc_flat, axis=1, keepdims=True) + 1e-6
    return (mfcc_flat - mean_value) / std_value

def audio_to_mfcc(y, sr, scaler=None, fit_scaler=False):
    """
    Convert audio to MFCC and normalize it with a shared training scaler when available.
    Input: y (audio array), sr (sample rate), scaler (saved scaler payload), fit_scaler (bool)
    Output: tuple (mfcc_normalized, scaler) or None if error
    """
    if len(y) < sr:
        return None, scaler

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, hop_length=HOP_LENGTH)

    if mfcc.shape[1] < IMG_WIDTH:
        pad_width = IMG_WIDTH - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode='constant')
    else:
        mfcc = mfcc[:, :IMG_WIDTH]

    mfcc_flat = mfcc.astype(np.float32).reshape(1, -1)
    mfcc_normalized_flat = apply_feature_scaler(mfcc_flat, scaler)
    mfcc_normalized = mfcc_normalized_flat.reshape(IMG_HEIGHT, IMG_WIDTH)
    return np.expand_dims(mfcc_normalized, axis=-1), scaler

def select_best_window(full_audio):
    if len(full_audio) <= TARGET_SAMPLES:
        return full_audio

    step_size = max(1, TARGET_SAMPLES // 4)
    best_start = 0
    best_energy = -1.0

    for start_idx in range(0, len(full_audio) - TARGET_SAMPLES + 1, step_size):
        window = full_audio[start_idx:start_idx + TARGET_SAMPLES]
        energy = float(np.mean(np.square(window)))
        if energy > best_energy:
            best_energy = energy
            best_start = start_idx

    return full_audio[best_start:best_start + TARGET_SAMPLES]


def load_audio_segment(file_path):
    """
    Load a representative segment for training.
    Prefer the highest-energy 4-second window so the model trains on stronger musical content,
    not silent or weak intro/outro sections.
    """
    try:
        y_full, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=30)
    except Exception:
        y_full, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION_SECONDS)

    if len(y_full) == 0:
        y = np.zeros(TARGET_SAMPLES, dtype=np.float32)
    else:
        y = select_best_window(y_full)

    if len(y) < TARGET_SAMPLES:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)), 'constant')
    else:
        y = y[:TARGET_SAMPLES]

    return y, sr


def blend_audio_segments(base_audio, extra_audio):
    if len(base_audio) != TARGET_SAMPLES or len(extra_audio) != TARGET_SAMPLES:
        return base_audio

    crossfade = min(COMPOSITE_CROSSFADE_SAMPLES, TARGET_SAMPLES // 4)
    midpoint = TARGET_SAMPLES // 2
    blend_start = max(0, midpoint - (crossfade // 2))
    blend_end = min(TARGET_SAMPLES, blend_start + crossfade)

    mixed = np.copy(base_audio)
    mixed[midpoint:] = extra_audio[midpoint:]

    if blend_end > blend_start:
        fade = np.linspace(0.0, 1.0, blend_end - blend_start, dtype=np.float32)
        mixed[blend_start:blend_end] = (
            (1.0 - fade) * base_audio[blend_start:blend_end]
            + fade * extra_audio[blend_start:blend_end]
        )

    peak = np.max(np.abs(mixed))
    if peak > 1.0:
        mixed = mixed / peak
    return mixed.astype(np.float32)


def build_composite_genre_spec(base_path, peer_paths, scaler=None):
    if not ENABLE_COMPOSITE_GENRE_AUGMENTATION or not peer_paths:
        return None

    candidate_paths = [path for path in peer_paths if path != base_path]
    if not candidate_paths:
        return None

    try:
        base_audio, sr = load_audio_segment(base_path)
        peer_path = random.choice(candidate_paths)
        peer_audio, _ = load_audio_segment(peer_path)
        composite_audio = blend_audio_segments(base_audio, peer_audio)
        composite_spec, _ = audio_to_mfcc(composite_audio, sr, scaler, False)
        return composite_spec
    except Exception:
        return None


def apply_mfcc_mask(spec):
    if spec is None:
        return None

    masked = np.array(spec, copy=True)
    freq_width = min(6, IMG_HEIGHT // 5)
    time_width = min(24, IMG_WIDTH // 6)
    freq_start = np.random.randint(0, max(1, IMG_HEIGHT - freq_width + 1))
    time_start = np.random.randint(0, max(1, IMG_WIDTH - time_width + 1))
    masked[freq_start:freq_start + freq_width, :, 0] = 0.0
    masked[:, time_start:time_start + time_width, 0] = 0.0
    return masked.astype(np.float32)


def load_and_augment(file_path, scaler=None, fit_scaler=False, augment=False, peer_paths=None, add_composite=False):
    """
    Load audio file and apply augmentation if requested
    Input: file_path (str), scaler (saved scaler payload), fit_scaler (bool), augment (bool)
    Output: tuple (list of spectrograms, scaler)
    """
    try:
        y, sr = load_audio_segment(file_path)
        specs = []

        spec, scaler = audio_to_mfcc(y, sr, scaler, fit_scaler)
        if spec is not None:
            specs.append(spec)

        if augment:
            y_noise = y + NOISE_FACTOR * np.random.randn(len(y))
            spec_noise, _ = audio_to_mfcc(y_noise, sr, scaler, False)
            if spec_noise is not None:
                specs.append(spec_noise)

            if ENABLE_TIME_STRETCH_AUGMENTATION:
                try:
                    rate = np.random.uniform(TIME_STRETCH_MIN, TIME_STRETCH_MAX)
                    y_stretch = librosa.effects.time_stretch(y, rate=rate)
                    if len(y_stretch) > TARGET_SAMPLES:
                        y_stretch = y_stretch[:TARGET_SAMPLES]
                    else:
                        y_stretch = np.pad(y_stretch, (0, TARGET_SAMPLES - len(y_stretch)), 'constant')
                    spec_stretch, _ = audio_to_mfcc(y_stretch, sr, scaler, False)
                    if spec_stretch is not None:
                        specs.append(spec_stretch)
                except Exception:
                    pass

            masked_spec = apply_mfcc_mask(spec)
            if masked_spec is not None:
                specs.append(masked_spec)

        if add_composite:
            for _ in range(COMPOSITE_VARIANTS):
                composite_spec = build_composite_genre_spec(file_path, peer_paths or [], scaler)
                if composite_spec is not None:
                    specs.append(composite_spec)

        return specs, scaler
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return [], scaler



## `build_robust_model` function
defines the CNN model architecture

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout,
    BatchNormalization, Input, GaussianNoise, LeakyReLU, Flatten
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import SparseCategoricalCrossentropy

# ---------------------------------------------------------
# MODEL ARCHITECTURE
# ---------------------------------------------------------
def build_robust_model(num_classes):
    """
    Build CNN model for genre classification.
    Input: num_classes (int)
    Output: compiled Keras model
    """
    reg = l2(L2_REGULARIZATION)

    model = Sequential()
    model.add(Input(shape=INPUT_SHAPE))
    model.add(GaussianNoise(0.10))

    model.add(Conv2D(24, (3, 5), padding='same', kernel_regularizer=reg))
    model.add(BatchNormalization())
    model.add(LeakyReLU(negative_slope=0.1))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.27))

    model.add(Conv2D(48, (3, 3), padding='same', kernel_regularizer=reg))
    model.add(BatchNormalization())
    model.add(LeakyReLU(negative_slope=0.1))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.33))

    model.add(Conv2D(96, (3, 3), padding='same', kernel_regularizer=reg))
    model.add(BatchNormalization())
    model.add(LeakyReLU(negative_slope=0.1))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.40))

    model.add(Flatten())
    model.add(Dense(128, kernel_regularizer=reg))
    model.add(LeakyReLU(negative_slope=0.1))
    model.add(Dropout(0.46))
    model.add(Dense(num_classes, activation='softmax'))

    optimizer = Adam(learning_rate=LEARNING_RATE)
    try:
        loss_fn = SparseCategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
    except TypeError:
        loss_fn = SparseCategoricalCrossentropy()
    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
    return model



## The Keras callback classes and related functions

In [ ]:
from tensorflow.keras.callbacks import Callback

# ---------------------------------------------------------
# CALLBACKS
# ---------------------------------------------------------
class GradioUpdateCallback(Callback):
    """
    Custom callback to update Gradio progress bar during training
    """
    def __init__(self, progress_obj, epochs):
        super().__init__()
        self.progress = progress_obj
        self.total_epochs = epochs

    def on_epoch_end(self, epoch, logs=None):
        percent = (epoch + 1) / self.total_epochs
        acc = logs.get('accuracy', 0)
        val_acc = logs.get('val_accuracy', 0)
        loss = logs.get('loss', 0)
        val_loss = logs.get('val_loss', 0)
        self.progress(
            percent,
            desc=f"Epoch {epoch+1}/{self.total_epochs} | Loss: {loss:.3f} | Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.2%}"
        )

class StopTrainingCallback(Callback):
    """
    Custom callback to allow user to stop training
    """
    def on_batch_end(self, batch, logs=None):
        global STOP_TRAINING_FLAG
        if STOP_TRAINING_FLAG:
            self.model.stop_training = True

def stop_training_func():
    """
    Set flag to stop training
    Input: None
    Output: status message (str)
    """
    global STOP_TRAINING_FLAG
    STOP_TRAINING_FLAG = True
    return "🛑 Stopping training at next checkpoint..."


##metrics and visualization functions

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# ---------------------------------------------------------
# METRICS AND VISUALIZATION
# ---------------------------------------------------------
def save_confusion_matrix(y_true, y_pred, save_path, class_names):
    cm = confusion_matrix(y_true, y_pred)

    fig_width = max(6, min(12, len(class_names) * 1.1))
    fig, ax = plt.subplots(figsize=(fig_width, fig_width))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)

    ax.set(
        xticks=list(range(len(class_names))),
        yticks=list(range(len(class_names))),
        xticklabels=class_names,
        yticklabels=class_names,
        title='Confusion Matrix',
        ylabel='True label',
        xlabel='Predicted label'
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')

    thresh = cm.max() / 2.0 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center', color='white' if cm[i, j] > thresh else 'black')

    fig.tight_layout()
    plt.savefig(save_path)
    plt.close()


def save_classification_report(y_true, y_pred, save_path, class_names):
    report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    with open(save_path, 'w') as f:
        f.write(report)


def plot_training_history(history):
    """
    Build a training summary plot for accuracy and loss.
    Returns a Matplotlib figure for the Gradio output panel.
    """
    history_data = getattr(history, 'history', {}) or {}
    epochs = range(1, len(history_data.get('loss', [])) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    train_acc = history_data.get('accuracy', [])
    val_acc = history_data.get('val_accuracy', [])
    train_loss = history_data.get('loss', [])
    val_loss = history_data.get('val_loss', [])

    axes[0].plot(epochs, train_acc, label='Train Accuracy', linewidth=2)
    if val_acc:
        axes[0].plot(epochs, val_acc, label='Validation Accuracy', linewidth=2)
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, train_loss, label='Train Loss', linewidth=2)
    if val_loss:
        axes[1].plot(epochs, val_loss, label='Validation Loss', linewidth=2)
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    fig.suptitle('BeatSense Training Summary', fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig



##train_logic function orchestrates the main training pipeline

In [ ]:
import os
import numpy as np
import datetime
import random
import pytz
import gradio as gr
import matplotlib.pyplot as plt

from sklearn.utils import class_weight
from sklearn.metrics import classification_report
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ---------------------------------------------------------
# TRAINING LOGIC
# ---------------------------------------------------------
def train_logic(progress=gr.Progress()):
    """
    Train the main model as a genre classifier.
    The final recommendation is computed later per user from the predicted genres.
    """
    global STOP_TRAINING_FLAG, global_model, global_scaler
    STOP_TRAINING_FLAG = False
    global_scaler = None

    def split_counts(split_name, genre_names):
        split_root = os.path.join(DATASET_PATH, split_name)
        counts = {}
        for genre in genre_names:
            genre_dir = os.path.join(split_root, genre)
            if not os.path.exists(genre_dir):
                counts[genre] = 0
                continue
            counts[genre] = sum(
                1 for file_name in os.listdir(genre_dir)
                if file_name.lower().endswith(AUDIO_EXTENSIONS)
            )
        return counts

    def label_summary(labels, genre_names):
        if len(labels) == 0:
            return 'empty'
        unique_labels, counts = np.unique(labels, return_counts=True)
        return ' | '.join(
            f"{genre_names[int(label)]}:{int(count)}"
            for label, count in zip(unique_labels, counts)
        )

    try:
        random.seed(SEED)
        np.random.seed(SEED)

        progress(0.03, desc='Validating dataset...')
        if not os.path.exists(DATASET_PATH):
            return 'Error: Dataset path not found!', None

        train_root = os.path.join(DATASET_PATH, 'train')
        genre_names = sorted(
            name for name in os.listdir(train_root)
            if os.path.isdir(os.path.join(train_root, name)) and name.lower() not in EXCLUDED_DATA_FOLDERS
        ) if os.path.exists(train_root) else []

        if not genre_names:
            return 'No genre folders were found under data/processed_data/train.', None

        print(f'Genres detected: {genre_names}')
        print(f'Training config: epochs={EPOCHS}, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}, l2={L2_REGULARIZATION}, label_smoothing={LABEL_SMOOTHING}')
        print(f'Augmentation config: noise={NOISE_FACTOR}, stretch={TIME_STRETCH_MIN}-{TIME_STRETCH_MAX}, augment_limit={AUGMENTATION_FILE_LIMIT}, composite_limit={COMPOSITE_AUGMENTATION_LIMIT}')
        split_count_map = {}
        for split_name in DATA_SPLITS:
            counts = split_counts(split_name, genre_names)
            split_count_map[split_name] = counts
            total = sum(counts.values())
            summary = ', '.join(f"{genre}:{count}" for genre, count in counts.items())
            print(f'{split_name.title()} split files: total={total} | {summary}')

        # Training and validation must contain every detected genre. Test data
        # remains optional because evaluation can safely fall back to validation.
        for required_split in ('train', 'val'):
            missing_genres = [
                genre for genre, count in split_count_map[required_split].items()
                if count == 0
            ]
            if missing_genres:
                missing_text = ', '.join(missing_genres)
                return f'Error: {required_split} is missing audio for: {missing_text}', None

        missing_test_genres = [
            genre for genre, count in split_count_map['test'].items() if count == 0
        ]
        if missing_test_genres:
            print(f"Test split is incomplete; validation fallback will be used where needed: {missing_test_genres}")

        label_map = {genre: idx for idx, genre in enumerate(genre_names)}
        X_train, y_train = [], []
        X_val, y_val = [], []
        X_test, y_test = [], []

        split_progress = {
            'train': (0.06, 0.42),
            'val': (0.44, 0.58),
            'test': (0.60, 0.70),
        }

        def load_split(split_name, target_specs, target_labels, augment):
            split_root = os.path.join(DATASET_PATH, split_name)
            genre_file_map = {}
            total_files = 0

            for genre in genre_names:
                genre_dir = os.path.join(split_root, genre)
                if not os.path.exists(genre_dir):
                    continue

                files = [
                    file_name for file_name in os.listdir(genre_dir)
                    if file_name.lower().endswith(AUDIO_EXTENSIONS)
                ]
                random.shuffle(files)
                files = files[:MAX_FILES_PER_LABEL]
                genre_file_map[genre] = files
                total_files += len(files)

            if total_files == 0:
                progress(split_progress[split_name][1], desc=f'No files found in {split_name}.')
                return 0

            processed_files = 0
            loaded_files = 0
            produced_specs = 0
            progress_start, progress_end = split_progress[split_name]
            print(f'Preparing {split_name}: {total_files} files across {len(genre_file_map)} genres')

            for genre_index, genre in enumerate(genre_names, start=1):
                files = genre_file_map.get(genre, [])
                if not files:
                    continue

                genre_dir = os.path.join(split_root, genre)
                print(f'Loading {split_name}/{genre}: {len(files)} files')

                for file_index, file_name in enumerate(files, start=1):
                    if STOP_TRAINING_FLAG:
                        return loaded_files

                    full_path = os.path.join(genre_dir, file_name)
                    is_valid, error = validate_audio_file(full_path)
                    if not is_valid:
                        print(f'Skipping {file_name}: {error}')
                        processed_files += 1
                        continue

                    use_augment = augment and file_index <= AUGMENTATION_FILE_LIMIT
                    use_composite = augment and file_index <= COMPOSITE_AUGMENTATION_LIMIT and len(files) > 1
                    peer_paths = [os.path.join(genre_dir, name) for name in files]
                    specs, _ = load_and_augment(
                        full_path,
                        scaler=None,
                        fit_scaler=False,
                        augment=use_augment,
                        peer_paths=peer_paths,
                        add_composite=use_composite,
                    )
                    for spec in specs:
                        target_specs.append(spec)
                        target_labels.append(label_map[genre])

                    loaded_files += 1
                    produced_specs += len(specs)
                    processed_files += 1

                    if processed_files == 1 or processed_files % PROGRESS_LOG_EVERY == 0 or processed_files == total_files:
                        progress_ratio = processed_files / total_files
                        progress_value = progress_start + (progress_ratio * (progress_end - progress_start))
                        progress(
                            progress_value,
                            desc=(
                                f'Loading {split_name}: {genre} '
                                f'({processed_files}/{total_files} files, genre {genre_index}/{len(genre_names)})...'
                            ),
                        )
                        print(
                            f'[{split_name}] {processed_files}/{total_files} files processed '
                            f'| current genre={genre} | latest file={file_name} | generated_specs={produced_specs}'
                        )

            print(f'Finished {split_name}: loaded_files={loaded_files}, generated_specs={produced_specs}')
            return loaded_files

        n_train = load_split('train', X_train, y_train, augment=True)
        if STOP_TRAINING_FLAG:
            return 'Training stopped by user.', None

        n_val = load_split('val', X_val, y_val, augment=False)
        if STOP_TRAINING_FLAG:
            return 'Training stopped by user.', None

        n_test = load_split('test', X_test, y_test, augment=False)

        if n_train == 0:
            return 'No training genre data found.', None
        if n_val == 0:
            return 'No validation genre data found.', None

        progress(0.74, desc='Converting features to arrays...')
        X_train = np.array(X_train, dtype=np.float32)
        y_train = np.array(y_train, dtype=np.int32)
        X_val = np.array(X_val, dtype=np.float32)
        y_val = np.array(y_val, dtype=np.int32)
        X_test = np.array(X_test, dtype=np.float32)
        y_test = np.array(y_test, dtype=np.int32)

        train_shape = X_train.shape
        val_shape = X_val.shape
        test_shape = X_test.shape
        train_flat = X_train.reshape(len(X_train), -1)
        feature_mean = train_flat.mean(axis=0)
        feature_std = train_flat.std(axis=0)
        feature_std = np.where(feature_std < 1e-6, 1.0, feature_std)

        X_train = ((train_flat - feature_mean) / feature_std).reshape(train_shape).astype(np.float32)
        X_val = ((X_val.reshape(len(X_val), -1) - feature_mean) / feature_std).reshape(val_shape).astype(np.float32)
        if len(X_test):
            X_test = ((X_test.reshape(len(X_test), -1) - feature_mean) / feature_std).reshape(test_shape).astype(np.float32)

        global_scaler = {
            'kind': 'tensor_stats',
            'mean': feature_mean.astype(np.float32).tolist(),
            'std': feature_std.astype(np.float32).tolist(),
        }
        with open(SCALER_PATH, 'wb') as scaler_file:
            pickle.dump(global_scaler, scaler_file)

        print(f'Training data shape: {X_train.shape}')
        print(f'Validation data shape: {X_val.shape}')
        print(f'Test data shape: {X_test.shape}')
        print(f'Train label distribution: {label_summary(y_train, genre_names)}')
        print(f'Validation label distribution: {label_summary(y_val, genre_names)}')
        print(f'Test label distribution: {label_summary(y_test, genre_names)}')
        print(f'Saved shared feature scaler to: {SCALER_PATH}')

        progress(0.78, desc='Building genre model...')
        global_model = build_robust_model(len(genre_names))
        print(f'Model built for {len(genre_names)} genres.')

        try:
            unique_train_labels = np.unique(y_train)
            class_weights = class_weight.compute_class_weight(
                class_weight='balanced',
                classes=unique_train_labels,
                y=y_train,
            )
            weights_dict = {int(label): float(weight) for label, weight in zip(unique_train_labels, class_weights)}
        except Exception:
            weights_dict = {idx: 1.0 for idx in range(len(genre_names))}
        print(f'Class weights: {weights_dict}')

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=LR_REDUCTION_FACTOR, patience=LR_REDUCTION_PATIENCE, verbose=1, min_lr=1e-6),
            GradioUpdateCallback(progress, EPOCHS),
            StopTrainingCallback(),
        ]

        progress(0.82, desc='Training genre classifier...')
        print('Starting model.fit()...')
        history = global_model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            class_weight=weights_dict,
            callbacks=callbacks,
            verbose=0,
        )

        if STOP_TRAINING_FLAG:
            return 'Training interrupted by user.', None

        progress(0.94, desc='Evaluating and saving outputs...')
        # EarlyStopping restores the weights with the lowest validation loss.
        # Measure those restored weights instead of reporting the final epoch gap.
        _, saved_train_accuracy = global_model.evaluate(X_train, y_train, batch_size=BATCH_SIZE, verbose=0)
        _, saved_val_accuracy = global_model.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)
        overfit_gap = float(saved_train_accuracy - saved_val_accuracy)

        eval_X = X_test if len(X_test) else X_val
        eval_y = y_test if len(y_test) else y_val
        eval_probabilities = global_model.predict(eval_X, verbose=0)
        eval_predictions = np.argmax(eval_probabilities, axis=1)
        eval_accuracy = float(np.mean(eval_predictions == eval_y)) if len(eval_y) else 0.0
        clipped_probabilities = np.clip(eval_probabilities, 1e-7, 1.0)
        normalized_entropy = -np.sum(
            clipped_probabilities * np.log(clipped_probabilities), axis=1
        ) / np.log(len(genre_names))
        mean_certainty = float(np.mean(1.0 - normalized_entropy)) if len(eval_y) else 0.0

        best_epoch = int(np.argmin(history.history['val_loss'])) + 1
        best_acc = float(saved_val_accuracy)
        print(f'Training summary: restored_train_acc={saved_train_accuracy:.4f}, restored_val_acc={saved_val_accuracy:.4f}, best_epoch={best_epoch}, eval_acc={eval_accuracy:.4f}, overfit_gap={overfit_gap:.4f}, mean_certainty={mean_certainty:.4f}')

        timestamp = datetime.datetime.now(pytz.timezone('Asia/Jerusalem')).strftime('%Y-%m-%d_%H-%M-%S')
        model_name = f'BeatSense_genre_{timestamp}_acc_{best_acc:.3f}_test_{eval_accuracy:.3f}_ep{best_epoch}.keras'
        save_path = os.path.join(HISTORY_DIR, model_name)
        global_model.save(save_path)
        scaler_name = f'{os.path.splitext(model_name)[0]}.scaler.pkl'
        scaler_save_path = os.path.join(HISTORY_DIR, scaler_name)
        with open(scaler_save_path, 'wb') as scaler_file:
            pickle.dump(global_scaler, scaler_file)
        print(f'Saved model to: {save_path}')
        print(f'Saved model scaler to: {scaler_save_path}')
        save_model_metadata(model_name, {
            'name': model_name,
            'task': 'genre',
            'genre_labels': genre_names,
            'scaler_file': scaler_name,
            'accuracy': float(f'{best_acc:.3f}'),
            'test_accuracy': float(f'{eval_accuracy:.3f}'),
            'epoch': best_epoch,
            'battles': [],
            'wins': 0,
            'losses': 0,
            'notes': {
                'train_samples': int(len(X_train)),
                'val_samples': int(len(X_val)),
                'test_samples': int(len(X_test)),
                'overfit_gap': float(f'{overfit_gap:.4f}'),
                'restored_train_accuracy': float(f'{saved_train_accuracy:.4f}'),
                'restored_val_accuracy': float(f'{saved_val_accuracy:.4f}'),
                'mean_prediction_certainty': float(f'{mean_certainty:.4f}'),
            },
        })

        report = classification_report(eval_y, eval_predictions, target_names=genre_names, zero_division=0)
        report_suffix = timestamp.replace('-', '').replace(':', '')
        report_path = os.path.join(METRICS_DIR, f'classification_report_{report_suffix}.txt')
        with open(report_path, 'w') as f:
            f.write(report)
        matrix_path = os.path.join(METRICS_DIR, f'confusion_matrix_{report_suffix}.png')
        save_confusion_matrix(eval_y, eval_predictions, matrix_path, genre_names)
        print(f'Saved evaluation report to: {report_path}')
        print(f'Saved confusion matrix to: {matrix_path}')

        fig = plot_training_history(history)
        progress(1.0, desc='Training complete.')

        status_message = f"""
Training complete.

Model saved to history:
- {model_name}

Validation accuracy: {best_acc:.1%}
Evaluation accuracy: {eval_accuracy:.1%}
Best epoch: {best_epoch}

Detected genres:
- {', '.join(genre_names)}

This model predicts music genres.
The final Like/Dislike recommendation is computed later from each user's preferences.
"""
        return status_message, fig
    except Exception as e:
        print(f'Training failed: {e}')
        return f'Error during training: {str(e)}', None



## 'Battle Arena' functionality (`get_score_for_file`, `download_youtube_audio`, `run_battle`, `resolve_winner`)



In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
try:
    import yt_dlp
except ImportError:
    yt_dlp = None

from tensorflow.keras.models import load_model

# ---------------------------------------------------------
# BATTLE ARENA - Model Comparison
# ---------------------------------------------------------
def get_score_for_file(model_name, file_path):
    """
    Get prediction score from a specific model
    Input: model_name (str), file_path (str)
    Output: float score or -1 on error
    """
    path = os.path.join(HISTORY_DIR, model_name)
    if not os.path.exists(path):
        return 0

    try:
        temp_model = load_model(path)

        if temp_model.input_shape[1:] != INPUT_SHAPE:
            return -1

        scaler = None
        if os.path.exists(SCALER_PATH):
            import pickle
            with open(SCALER_PATH, 'rb') as f:
                scaler = pickle.load(f)

        specs, _ = load_and_augment(file_path, scaler=scaler, fit_scaler=False, augment=False)
        if not specs:
            return 0

        preds = temp_model.predict(np.array(specs), verbose=0)
        return float(np.mean(preds))

    except Exception as e:
        print(f"Error in get_score_for_file: {e}")
        return 0

def download_youtube_audio(url, output_path):
    """
    Download audio from YouTube URL
    Input: url (str), output_path (str)
    Output: tuple (success: bool, message: str)
    """
    if not validate_url(url):
        return False, "Invalid YouTube URL"

    if os.path.exists(output_path):
        os.remove(output_path)

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': output_path.replace('.wav', ''),
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav'
        }],
        'quiet': True,
        'no_warnings': True
    }

    if yt_dlp is None:
        return False, 'yt-dlp is not installed in this runtime.'

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        return True, "Download successful"
    except Exception as e:
        return False, f"Download error: {str(e)}"

def run_battle(model_a, model_b, file, url):
    if not model_a or not model_b:
        return "Please select 2 models to compare.", None, None

    if model_a == model_b:
        return "Please select different models.", None, None

    target = file
    if url and url.strip():
        dl_path = os.path.join(BASE_PROJECT_PATH, "battle_temp.wav")
        success, message = download_youtube_audio(url, dl_path)
        if not success:
            return f"Error: {message}", None, None
        target = dl_path

    if not target:
        return "Please provide audio file or YouTube URL.", None, None

    is_valid, error = validate_audio_file(target)
    if not is_valid:
        return f"Error: {error}", None, None

    score_a = get_score_for_file(model_a, target)
    score_b = get_score_for_file(model_b, target)

    if score_a == -1 or score_b == -1:
        return "One of the models is incompatible. Please delete old models in Options tab.", None, None

    battle_data = [model_a, score_a, model_b, score_b]
    pred_a = "LIKE" if score_a > 0.5 else "DISLIKE"
    pred_b = "LIKE" if score_b > 0.5 else "DISLIKE"

    result_text = f"""
Battle Results

Model A: {model_a}
- Score: {score_a:.1%}
- Prediction: {pred_a}

Model B: {model_b}
- Score: {score_b:.1%}
- Prediction: {pred_b}

Higher score means stronger prediction for LIKE.
Judging the battle updates battle history only.
Promoting a winner to best_model.keras stays manual.
"""

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(['Model A', 'Model B'], [score_a, score_b], color=['#3498db', '#e74c3c'], alpha=0.8)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Confidence Score', fontsize=12)
    ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Decision Threshold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height, f'{height:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

    plt.tight_layout()
    return result_text, fig, battle_data

def resolve_winner(label_truth, battle_data):
    """
    Determine winner based on ground truth.
    This does not copy the winner into best_model.keras automatically.
    """
    if not battle_data:
        return "Please run a battle first."

    name_a, score_a, name_b, score_b = battle_data

    if label_truth == "LIKE":
        winner = name_a if score_a > score_b else name_b
        winner_score = max(score_a, score_b)
    else:
        winner = name_a if score_a < score_b else name_b
        winner_score = min(score_a, score_b)

    src = os.path.join(HISTORY_DIR, winner)
    if not os.path.exists(src):
        return "Winner model file not found."

    return f"""
Winner: {winner}

Battle history was updated.
Winner score: {winner_score:.1%}
Ground truth: {label_truth}

The admin still needs to choose manually if this winner should become best_model.keras.
Use Model Management to promote it.
"""


##`predict_one` function handles single audio analysis and prediction

In [ ]:
import os
import json
import numpy as np
import librosa

from tensorflow.keras.models import load_model

# ---------------------------------------------------------
# PREDICTION - Single Audio Analysis
# ---------------------------------------------------------
def predict_one(audio_path, url):
    """
    Predict matching genres for a single audio file.
    The personal recommendation is applied later per user.
    """
    target = audio_path
    if url and url.strip():
        if not validate_url(url):
            return 'Invalid YouTube URL format'

        dl_path = os.path.join(BASE_PROJECT_PATH, 'temp_pred.wav')
        success, message = download_youtube_audio(url, dl_path)
        if not success:
            return f'Error: {message}'
        target = dl_path

    if not target:
        return 'Please provide audio file or YouTube URL'

    is_valid, error = validate_audio_file(target)
    if not is_valid:
        return f'Error: {error}'

    if not os.path.exists(BEST_MODEL_PATH):
        return 'No trained model available. Please train a model first.'

    metadata_path = BEST_MODEL_PATH.replace('.keras', '.json')
    metadata = {}
    if os.path.exists(metadata_path):
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)

    genre_labels = metadata.get('genre_labels', [])
    if metadata.get('task') != 'genre' or not genre_labels:
        return 'The active best model is not a genre-classification model yet.'

    try:
        model = load_model(BEST_MODEL_PATH)
        scaler_payload = None
        if os.path.exists(SCALER_PATH):
            with open(SCALER_PATH, 'rb') as scaler_file:
                scaler_payload = pickle.load(scaler_file)
        y_full, sr = librosa.load(target, sr=SAMPLE_RATE)
        segment_predictions = []

        for t in MULTI_SEGMENT_TIMESTAMPS:
            start_idx = int(max(0, len(y_full) - TARGET_SAMPLES) * t)
            segment = y_full[start_idx:start_idx + TARGET_SAMPLES]
            if len(segment) < TARGET_SAMPLES:
                segment = np.pad(segment, (0, TARGET_SAMPLES - len(segment)), 'constant')
            spec, _ = audio_to_mfcc(segment, sr, scaler=scaler_payload, fit_scaler=False)
            if spec is None:
                continue
            pred = model.predict(np.expand_dims(spec, axis=0), verbose=0)[0]
            segment_predictions.append(pred)

        if not segment_predictions:
            return 'Audio too short or could not process segments.'

        mean_predictions = np.mean(np.array(segment_predictions), axis=0)
        top_indices = np.argsort(mean_predictions)[::-1][:3]
        top_lines = [f'- {genre_labels[idx]}: {mean_predictions[idx]:.1%}' for idx in top_indices]
        primary_idx = int(top_indices[0])

        return f"""
BeatSense Genre Analysis

Primary genre: {genre_labels[primary_idx]}
Confidence: {mean_predictions[primary_idx]:.1%}

Top matching styles:
{chr(10).join(top_lines)}

This result should be compared with each user's liked and disliked genres.
"""
    except Exception as e:
        return f'Error during prediction: {str(e)}'


##general utility functions (`get_models_list`, `delete_all_models`, `refresh_list`)

In [ ]:
import os
import gradio as gr
import shutil # For delete_all_models

# ---------------------------------------------------------
# UTILITY FUNCTIONS
# ---------------------------------------------------------
def get_models_list():
    """
    Get list of all saved models
    Input: None
    Output: list of model filenames
    """
    if not os.path.exists(HISTORY_DIR):
        return []
    return sorted(
        [f for f in os.listdir(HISTORY_DIR) if f.endswith(".keras")],
        reverse=True
    )

def delete_all_models():
    """
    Delete all model history and reset
    Input: None
    Output: status message (str)
    """
    if not os.path.exists(HISTORY_DIR):
        return "⚠️ History folder not found."

    count = 0
    for f in os.listdir(HISTORY_DIR):
        os.remove(os.path.join(HISTORY_DIR, f))
        count += 1

    if os.path.exists(BEST_MODEL_PATH):
        os.remove(BEST_MODEL_PATH)

    if os.path.exists(SCALER_PATH):
        os.remove(SCALER_PATH)

    return f"🗑️ Deleted {count} models. Clean slate! Start fresh training now."

def refresh_list():
    """
    Refresh model dropdown lists
    Input: None
    Output: tuple of Gradio update objects
    """
    models = get_models_list()
    return gr.update(choices=models), gr.update(choices=models)


## all functions related to model metadata management (`get_model_metadata`, `save_model_metadata`, `update_battle_history`, `set_active_model`, `delete_specific_model`, `get_model_list_html`)



In [ ]:
import os
import json
import shutil
import datetime

from tensorflow.keras.models import load_model

# ---------------------------------------------------------
# MODEL METADATA MANAGEMENT
# ---------------------------------------------------------
def get_model_metadata(model_name):
    metadata_path = os.path.join(HISTORY_DIR, f"{model_name.replace('.keras', '')}.json")
    if os.path.exists(metadata_path):
        with open(metadata_path, 'r') as f:
            return json.load(f)

    try:
        parts = model_name.replace('.keras', '').split('_')
        accuracy = 'N/A'
        epoch = 'N/A'
        for part in parts:
            if 'acc' in part and len(part) > 3:
                try:
                    accuracy = float(part.replace('acc', ''))
                except ValueError:
                    pass
            elif 'ep' in part and len(part) > 2:
                try:
                    epoch = int(part.replace('ep', ''))
                except ValueError:
                    pass
        return {
            'name': model_name,
            'task': 'unknown',
            'accuracy': accuracy,
            'epoch': epoch,
            'battles': [],
            'wins': 0,
            'losses': 0,
        }
    except Exception as e:
        print(f'Error parsing metadata from filename {model_name}: {e}')
        return {
            'name': model_name,
            'task': 'unknown',
            'accuracy': 'N/A',
            'epoch': 'N/A',
            'battles': [],
            'wins': 0,
            'losses': 0,
        }

def save_model_metadata(model_name, metadata):
    metadata_path = os.path.join(HISTORY_DIR, f"{model_name.replace('.keras', '')}.json")
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)

def update_battle_history(winner_name, loser_name):
    winner_meta = get_model_metadata(winner_name)
    loser_meta = get_model_metadata(loser_name)
    battle_timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

    winner_meta['wins'] = winner_meta.get('wins', 0) + 1
    winner_meta.setdefault('battles', []).append({'timestamp': battle_timestamp, 'result': 'win', 'opponent': loser_name})
    loser_meta['losses'] = loser_meta.get('losses', 0) + 1
    loser_meta.setdefault('battles', []).append({'timestamp': battle_timestamp, 'result': 'loss', 'opponent': winner_name})

    save_model_metadata(winner_name, winner_meta)
    save_model_metadata(loser_name, loser_meta)

def set_active_model(model_name):
    src = os.path.join(HISTORY_DIR, model_name)
    if not os.path.exists(src):
        return 'Model file not found.'

    try:
        shutil.copy(src, BEST_MODEL_PATH)
        src_meta = os.path.join(HISTORY_DIR, f"{model_name.replace('.keras', '')}.json")
        dst_meta = BEST_MODEL_PATH.replace('.keras', '.json')
        if os.path.exists(src_meta):
            shutil.copy(src_meta, dst_meta)
        elif os.path.exists(dst_meta):
            os.remove(dst_meta)
        global global_model
        global_model = load_model(BEST_MODEL_PATH)
        return f'Model {model_name} is now the active best model.'
    except Exception as e:
        return f'Error: {str(e)}'

def delete_specific_model(model_name):
    model_path = os.path.join(HISTORY_DIR, model_name)
    metadata_path = os.path.join(HISTORY_DIR, f"{model_name.replace('.keras', '')}.json")
    try:
        if os.path.exists(model_path):
            os.remove(model_path)
        if os.path.exists(metadata_path):
            os.remove(metadata_path)
        return f'Deleted {model_name}'
    except Exception as e:
        return f'Error: {str(e)}'

def get_model_list_html():
    models = get_models_list()
    if not models:
        return '<p>No models found. Train a model first.</p>'

    html = "<div style='font-family: Arial, sans-serif;'>"
    for model in models:
        meta = get_model_metadata(model)
        wins = meta.get('wins', 0)
        losses = meta.get('losses', 0)
        total_battles = wins + losses
        win_rate = (wins / total_battles * 100) if total_battles > 0 else 0
        task = meta.get('task', 'unknown')
        html += f"""
        <div style='padding: 15px; margin: 10px 0; border-radius: 5px;'>
            <h3 style='margin: 0 0 10px 0; font-family: Arial, sans-serif;'>{model}</h3>
            <p style='margin: 5px 0; font-family: Arial, sans-serif;'><strong>Task:</strong> {task}</p>
            <p style='margin: 5px 0; font-family: Arial, sans-serif;'><strong>Accuracy:</strong> {meta.get('accuracy', 'N/A')}</p>
            <p style='margin: 5px 0; font-family: Arial, sans-serif;'><strong>Battles:</strong> {total_battles} (W: {wins}, L: {losses}, Win Rate: {win_rate:.1f}%)</p>
        </div>
        """
    html += '</div>'
    return html


## Gradio interface setup, which defines the entire web application layout and interactivity



In [ ]:
import gradio as gr
import os # for os.path.exists in the Gradio UI section for button visibility

# ---------------------------------------------------------
# GRADIO INTERFACE
# ---------------------------------------------------------
custom_css = """
* {
    font-family: Arial, sans-serif !important;
}

.gradio-container {
    font-family: Arial, sans-serif !important;
}
"""

with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:
    gr.Markdown("""
    # ♩ BeatSense - AI Music Taste Classifier
    ### Train AI to understand your music preferences using advanced MFCC analysis
    """)

    battle_state = gr.State()
    predict_state = gr.State({"stage": "upload"})  # upload or results

    with gr.Tabs():
        # --- TAB 1: TRAIN ---
        with gr.Tab("🎓 Train Model"):
            gr.Markdown("""
            ### Training Process
            1. Loads audio files from your dataset
            2. Extracts MFCC features with proper normalization
            3. Trains a CNN model with regularization
            4. Saves a new timestamped model to model history
            """)

            with gr.Row():
                btn_train = gr.Button("🚀 Start Training", variant="primary", size="lg")
                btn_stop = gr.Button("🛑 Stop Training", variant="stop", size="lg")

            status_box = gr.Markdown(label="Training Status")
            plot_box = gr.Plot(label="Training Metrics")

            btn_train.click(train_logic, outputs=[status_box, plot_box])
            btn_stop.click(stop_training_func, outputs=status_box)

        # --- TAB 2: ARENA ---
        with gr.Tab("⚔️ Model Arena"):
            arena_stage = gr.State("setup")  # setup or results

            # Setup screen
            with gr.Group(visible=True) as arena_setup:
                gr.Markdown("### Select Models to Battle")

                with gr.Row():
                    mod_a = gr.Dropdown(
                        label="🔵 Model A",
                        choices=get_models_list(),
                        info="Select first model"
                    )
                    mod_b = gr.Dropdown(
                        label="🔴 Model B",
                        choices=get_models_list(),
                        info="Select second model"
                    )

                ref_btn = gr.Button("🔄 Refresh Model List", size="sm")

                gr.Markdown("### Upload Audio or Provide YouTube Link")
                with gr.Row():
                    bat_audio = gr.Audio(label="Audio File", type="filepath")
                    bat_url = gr.Textbox(
                        label="YouTube URL",
                        placeholder="https://www.youtube.com/watch?v=...",
                        info="Alternative to file upload"
                    )

                fight_btn = gr.Button("⚔️ START BATTLE!", variant="primary", size="lg")

            # Results screen
            with gr.Group(visible=False) as arena_results:
                bat_res = gr.Markdown(label="Battle Results")
                bat_plot = gr.Plot(label="Score Comparison")

                gr.Markdown("### What is the correct label for this song?")

                with gr.Row():
                    btn_real_like = gr.Button("👍 I LIKE This Song", variant="primary", size="lg")
                    btn_real_dis = gr.Button("👎 I DISLIKE This Song", variant="secondary", size="lg")

                judge_res = gr.Markdown(label="Winner")
                btn_back_arena = gr.Button("↩️ New Battle", size="lg")

            def show_battle_results(model_a, model_b, file, url):
                result_text, fig, battle_data = run_battle(model_a, model_b, file, url)
                return (
                    gr.update(visible=False),  # hide setup
                    gr.update(visible=True),   # show results
                    result_text,
                    fig,
                    battle_data
                )

            def reset_arena():
                return (
                    gr.update(visible=True),   # show setup
                    gr.update(visible=False),  # hide results
                    "",  # clear results
                    None,  # clear plot
                    ""   # clear judge
                )

            def resolve_and_record(label_truth, battle_data):
                result = resolve_winner(label_truth, battle_data)
                if battle_data and "Winner:" in result:
                    name_a, score_a, name_b, score_b = battle_data
                    if label_truth == "LIKE":
                        winner = name_a if score_a > score_b else name_b
                        loser = name_b if score_a > score_b else name_a
                    else:
                        winner = name_a if score_a < score_b else name_b
                        loser = name_b if score_a < score_b else name_a
                    update_battle_history(winner, loser)
                return result

            ref_btn.click(refresh_list, outputs=[mod_a, mod_b])

            fight_btn.click(
                show_battle_results,
                inputs=[mod_a, mod_b, bat_audio, bat_url],
                outputs=[arena_setup, arena_results, bat_res, bat_plot, battle_state]
            )

            btn_real_like.click(
                fn=lambda s: resolve_and_record("LIKE", s),
                inputs=battle_state,
                outputs=judge_res
            )
            btn_real_dis.click(
                fn=lambda s: resolve_and_record("DISLIKE", s),
                inputs=battle_state,
                outputs=judge_res
            )

            btn_back_arena.click(
                reset_arena,
                outputs=[arena_setup, arena_results, bat_res, bat_plot, judge_res]
            )

               # --- TAB 3: PREDICT ---
        with gr.Tab("🔎 Predict"):
            # Upload screen
            with gr.Group(visible=True) as predict_upload:
                gr.Markdown("### Analyze New Music")
                gr.Markdown("Upload an audio file or provide a YouTube link to get AI predictions.")

                with gr.Row():
                    in_audio = gr.Audio(label="Upload Audio File", type="filepath")
                    in_url = gr.Textbox(
                        label="Or Enter YouTube URL",
                        placeholder="https://www.youtube.com/watch?v=...",
                        info="Paste YouTube link"
                    )

                pred_error = gr.Markdown(value="")  # NEW: error box in upload screen
                btn_pred = gr.Button("🎯 Analyze Music", variant="primary", size="lg")

            # Results screen
            with gr.Group(visible=False) as predict_results:
                out_res = gr.Markdown(label="Analysis Results")
                btn_back_pred = gr.Button("↩️ Analyze Another Song", size="lg")

            def show_prediction_results(audio_path, url):
                # default UI state: stay on upload unless success
                # clear previous error
                if (not audio_path) and (not (url and url.strip())):
                    return (
                        gr.update(visible=True),    # keep upload visible
                        gr.update(visible=False),   # keep results hidden
                        "",                         # clear results text
                        "❌ Please provide audio file or YouTube URL"
                    )

                # If URL provided - validate and download
                target = audio_path
                if url and url.strip():
                    if not validate_url(url):
                        return (
                            gr.update(visible=True),
                            gr.update(visible=False),
                            "",
                            "❌ Invalid YouTube URL format"
                        )

                    dl_path = os.path.join(BASE_PROJECT_PATH, "temp_pred.wav")
                    success, message = download_youtube_audio(url, dl_path)
                    if not success:
                        return (
                            gr.update(visible=True),
                            gr.update(visible=False),
                            "",
                            f"❌ {message}"
                        )
                    target = dl_path

                # Validate local file
                is_valid, error = validate_audio_file(target)
                if not is_valid:
                    return (
                        gr.update(visible=True),
                        gr.update(visible=False),
                        "",
                        f"❌ {error}"
                    )

                # Run prediction
                result = predict_one(target, "")

                # Show results screen on success
                return (
                    gr.update(visible=False),  # hide upload
                    gr.update(visible=True),   # show results
                    result,                    # results text
                    ""                         # clear error
                )

            def reset_predict():
                # Go back to upload screen and clear inputs/outputs
                return (
                    gr.update(visible=True),   # show upload
                    gr.update(visible=False),  # hide results
                    None,                      # clear audio component
                    "",                        # clear url textbox
                    "",                        # clear results markdown
                    ""                         # clear error markdown
                )

            # Note: outputs now include in_audio, in_url, out_res, pred_error
            btn_pred.click(
                show_prediction_results,
                inputs=[in_audio, in_url],
                outputs=[predict_upload, predict_results, out_res, pred_error]
            )

            btn_back_pred.click(
                reset_predict,
                outputs=[predict_upload, predict_results, in_audio, in_url, out_res, pred_error]
            )


        # --- TAB 4: SETTINGS ---
        with gr.Tab("⚙️ Settings"):
            gr.Markdown("### Model Management")

            # Model list view
            model_list_html = gr.HTML(label="Saved Models")

            # Model selector for actions
            gr.Markdown("### Model Actions")
            selected_model = gr.Dropdown(
                label="Select Model",
                choices=get_models_list(),
                info="Choose a model to view details or perform actions"
            )

            refresh_models_btn = gr.Button("🔄 Refresh List", size="sm")

            # Model details
            model_details = gr.Markdown(label="Model Details")

            # Action buttons
            with gr.Row():
                set_active_btn = gr.Button("✅ Set as Active Model", variant="primary")
                delete_model_btn = gr.Button("🗑️ Delete This Model", variant="stop")

            action_result = gr.Markdown(label="Result")

            gr.Markdown("---")
            gr.Markdown("### Danger Zone")
            gr.Markdown("**Warning:** This will delete ALL models and reset the system.")

            del_all_btn = gr.Button("🗑️ Delete All Models", variant="stop")
            del_all_msg = gr.Markdown(label="Status")

            def show_model_details(model_name):
                if not model_name:
                    return "Select a model to view details"

                meta = get_model_metadata(model_name)
                wins = meta.get("wins", 0)
                losses = meta.get("losses", 0)
                total = wins + losses
                win_rate = (wins / total * 100) if total > 0 else 0

                battles_text = ""
                if meta.get("battles"):
                    battles_text = "\n\n**Recent Battles:**\n"
                    for b in meta["battles"][-5:]:  # Last 5 battles
                        emoji = "🏆" if b["result"] == "win" else "❌"
                        battles_text += f"- {emoji} {b['result'].upper()} vs {b['opponent']} ({b['timestamp']})\n"

                return f"""
## {model_name}

**Validation Accuracy:** {meta.get('accuracy', 'N/A')}
**Training Epoch:** {meta.get('epoch', 'N/A')}

**Battle Statistics:**
- Total Battles: {total}
- Wins: {wins}
- Losses: {losses}
- Win Rate: {win_rate:.1f}%
{battles_text}
"""

            def refresh_all_models():
                models = get_models_list()
                html = get_model_list_html()
                return gr.update(choices=models), html

            # Load initial model list
            demo.load(
                fn=lambda: get_model_list_html(),
                outputs=model_list_html
            )

            refresh_models_btn.click(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

            selected_model.change(
                show_model_details,
                inputs=selected_model,
                outputs=model_details
            )

            set_active_btn.click(
                set_active_model,
                inputs=selected_model,
                outputs=action_result
            )

            delete_model_btn.click(
                delete_specific_model,
                inputs=selected_model,
                outputs=action_result
            ).then(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

            del_all_btn.click(delete_all_models, outputs=del_all_msg).then(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

        # --- TAB 5: CONFIGURATION ---
        with gr.Tab("📋 Configuration"):
            gr.Markdown(f"""
            ### Current System Configuration

            All settings are defined as constants in the code. To modify them, edit the source file.

            #### Model Architecture
            - **Input Shape:** MFCC {IMG_HEIGHT}x{IMG_WIDTH}
            - **Architecture:** 3 Convolutional blocks + Dense layers
            - **L2 Regularization:** {L2_REGULARIZATION}
            - **Dropout Rates:** 0.3-0.5
            - **Activation:** LeakyReLU (alpha=0.1)

            #### Training Parameters
            - **Batch Size:** {BATCH_SIZE}
            - **Epochs:** {EPOCHS}
            - **Learning Rate:** {LEARNING_RATE}
            - **Label Smoothing:** {LABEL_SMOOTHING}
            - **Early Stopping Patience:** {EARLY_STOPPING_PATIENCE} epochs
            - **LR Reduction Patience:** {LR_REDUCTION_PATIENCE} epochs
            - **LR Reduction Factor:** {LR_REDUCTION_FACTOR}

            #### Audio Processing
            - **Sample Rate:** {SAMPLE_RATE} Hz
            - **Segment Duration:** {DURATION_SECONDS} seconds
            - **MFCC Bands:** {N_MFCC}
            - **Hop Length:** {HOP_LENGTH}

            #### Data Augmentation
            - **Noise Factor:** {NOISE_FACTOR}
            - **Time Stretch Range:** {TIME_STRETCH_MIN} - {TIME_STRETCH_MAX}

            #### Prediction
            - **Multi-Segment Analysis:** {len(MULTI_SEGMENT_TIMESTAMPS)} segments
            - **Segment Positions:** {', '.join([f'{int(t*100)}%' for t in MULTI_SEGMENT_TIMESTAMPS])}
            - **Average Weight:** {SEGMENT_WEIGHT_AVG}
            - **Max Weight:** {SEGMENT_WEIGHT_MAX}

            #### File Limits
            - **Max File Size:** {MAX_FILE_SIZE_MB} MB
            """)

if __name__ == "__main__":
    if os.environ.get("BEATSENSE_AUTORUN_MAIN_TRAINING") == "1":
        print("Skipping Gradio launch during website-triggered main training.")
    else:
        demo.launch()

print("Gradio interface defined.")

/tmp/ipykernel_1296/2076286469.py:17: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:
/tmp/ipykernel_1296/2076286469.py:17: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ef895701d468ad18ab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio interface defined.


**Reasoning**:
The previous execution produced DeprecationWarnings regarding the 'theme' and 'css' parameters in `gr.Blocks()`. These should now be passed to `demo.launch()` according to Gradio's updated API. I will modify the last code block to move these parameters.



In [ ]:
import gradio as gr
import os # for os.path.exists in the Gradio UI section for button visibility

# ---------------------------------------------------------
# GRADIO INTERFACE
# ---------------------------------------------------------
custom_css = """
* {
    font-family: Arial, sans-serif !important;
}

.gradio-container {
    font-family: Arial, sans-serif !important;
}
"""

with gr.Blocks(title="BeatSense - AI Music Classifier") as demo:
    gr.Markdown("""
    # ♩&#xFE0F; BeatSense - AI Music Taste Classifier
    ### Train AI to understand your music preferences using advanced MFCC analysis
    """)

    battle_state = gr.State()
    predict_state = gr.State({"stage": "upload"})  # upload or results

    with gr.Tabs():
        # --- TAB 1: TRAIN ---
        with gr.Tab("🎓 Train Model"):
            gr.Markdown("""
            ### Training Process
            1. Loads audio files from your dataset
            2. Extracts MFCC features with proper normalization
            3. Trains a CNN model with regularization
            4. Saves a new timestamped model to model history
            """)

            with gr.Row():
                btn_train = gr.Button("🚀 Start Training", variant="primary", size="lg")
                btn_stop = gr.Button("🛑 Stop Training", variant="stop", size="lg")

            status_box = gr.Markdown(label="Training Status")
            plot_box = gr.Plot(label="Training Metrics")

            btn_train.click(train_logic, outputs=[status_box, plot_box])
            btn_stop.click(stop_training_func, outputs=status_box)

        # --- TAB 2: ARENA ---
        with gr.Tab("⚔️ Model Arena"):
            arena_stage = gr.State("setup")  # setup or results

            # Setup screen
            with gr.Group(visible=True) as arena_setup:
                gr.Markdown("### Select Models to Battle")

                with gr.Row():
                    mod_a = gr.Dropdown(
                        label="🔵 Model A",
                        choices=get_models_list(),
                        info="Select first model"
                    )
                    mod_b = gr.Dropdown(
                        label="🔴 Model B",
                        choices=get_models_list(),
                        info="Select second model"
                    )

                ref_btn = gr.Button("🔄 Refresh Model List", size="sm")

                gr.Markdown("### Upload Audio or Provide YouTube Link")
                with gr.Row():
                    bat_audio = gr.Audio(label="Audio File", type="filepath")
                    bat_url = gr.Textbox(
                        label="YouTube URL",
                        placeholder="https://www.youtube.com/watch?v=...",
                        info="Alternative to file upload"
                    )

                fight_btn = gr.Button("⚔️ START BATTLE!", variant="primary", size="lg")

            # Results screen
            with gr.Group(visible=False) as arena_results:
                bat_res = gr.Markdown(label="Battle Results")
                bat_plot = gr.Plot(label="Score Comparison")

                gr.Markdown("### What is the correct label for this song?")

                with gr.Row():
                    btn_real_like = gr.Button("👍 I LIKE This Song", variant="primary", size="lg")
                    btn_real_dis = gr.Button("👎 I DISLIKE This Song", variant="secondary", size="lg")

                judge_res = gr.Markdown(label="Winner")
                btn_back_arena = gr.Button("↩️ New Battle", size="lg")

            def show_battle_results(model_a, model_b, file, url):
                result_text, fig, battle_data = run_battle(model_a, model_b, file, url)
                return (
                    gr.update(visible=False),  # hide setup
                    gr.update(visible=True),   # show results
                    result_text,
                    fig,
                    battle_data
                )

            def reset_arena():
                return (
                    gr.update(visible=True),   # show setup
                    gr.update(visible=False),  # hide results
                    "",  # clear results
                    None,  # clear plot
                    ""   # clear judge
                )

            def resolve_and_record(label_truth, battle_data):
                result = resolve_winner(label_truth, battle_data)
                if battle_data and "Winner:" in result:
                    name_a, score_a, name_b, score_b = battle_data
                    if label_truth == "LIKE":
                        winner = name_a if score_a > score_b else name_b
                        loser = name_b if score_a > score_b else name_a
                    else:
                        winner = name_a if score_a < score_b else name_b
                        loser = name_b if score_a < score_b else name_a
                    update_battle_history(winner, loser)
                return result

            ref_btn.click(refresh_list, outputs=[mod_a, mod_b])

            fight_btn.click(
                show_battle_results,
                inputs=[mod_a, mod_b, bat_audio, bat_url],
                outputs=[arena_setup, arena_results, bat_res, bat_plot, battle_state]
            )

            btn_real_like.click(
                fn=lambda s: resolve_and_record("LIKE", s),
                inputs=battle_state,
                outputs=judge_res
            )
            btn_real_dis.click(
                fn=lambda s: resolve_and_record("DISLIKE", s),
                inputs=battle_state,
                outputs=judge_res
            )

            btn_back_arena.click(
                reset_arena,
                outputs=[arena_setup, arena_results, bat_res, bat_plot, judge_res]
            )

               # --- TAB 3: PREDICT ---
        with gr.Tab("🔎 Predict"):
            # Upload screen
            with gr.Group(visible=True) as predict_upload:
                gr.Markdown("### Analyze New Music")
                gr.Markdown("Upload an audio file or provide a YouTube link to get AI predictions.")

                with gr.Row():
                    in_audio = gr.Audio(label="Upload Audio File", type="filepath")
                    in_url = gr.Textbox(
                        label="Or Enter YouTube URL",
                        placeholder="https://www.youtube.com/watch?v=...",
                        info="Paste YouTube link"
                    )

                pred_error = gr.Markdown(value="")  # NEW: error box in upload screen
                btn_pred = gr.Button("🎯 Analyze Music", variant="primary", size="lg")

            # Results screen
            with gr.Group(visible=False) as predict_results:
                out_res = gr.Markdown(label="Analysis Results")
                btn_back_pred = gr.Button("↩️ Analyze Another Song", size="lg")

            def show_prediction_results(audio_path, url):
                # default UI state: stay on upload unless success
                # clear previous error
                if (not audio_path) and (not (url and url.strip())):
                    return (
                        gr.update(visible=True),    # keep upload visible
                        gr.update(visible=False),   # keep results hidden
                        "",                         # clear results text
                        "❌ Please provide audio file or YouTube URL"
                    )

                # If URL provided - validate and download
                target = audio_path
                if url and url.strip():
                    if not validate_url(url):
                        return (
                            gr.update(visible=True),
                            gr.update(visible=False),
                            "",
                            "❌ Invalid YouTube URL format"
                        )

                    dl_path = os.path.join(BASE_PROJECT_PATH, "temp_pred.wav")
                    success, message = download_youtube_audio(url, dl_path)
                    if not success:
                        return (
                            gr.update(visible=True),
                            gr.update(visible=False),
                            "",
                            f"❌ {message}"
                        )
                    target = dl_path

                # Validate local file
                is_valid, error = validate_audio_file(target)
                if not is_valid:
                    return (
                        gr.update(visible=True),
                        gr.update(visible=False),
                        "",
                        f"❌ {error}"
                    )

                # Run prediction
                result = predict_one(target, "")

                # Show results screen on success
                return (
                    gr.update(visible=False),  # hide upload
                    gr.update(visible=True),   # show results
                    result,                    # results text
                    ""                         # clear error
                )

            def reset_predict():
                # Go back to upload screen and clear inputs/outputs
                return (
                    gr.update(visible=True),   # show upload
                    gr.update(visible=False),  # hide results
                    None,                      # clear audio component
                    "",                        # clear url textbox
                    "",                        # clear results markdown
                    ""                         # clear error markdown
                )

            # Note: outputs now include in_audio, in_url, out_res, pred_error
            btn_pred.click(
                show_prediction_results,
                inputs=[in_audio, in_url],
                outputs=[predict_upload, predict_results, out_res, pred_error]
            )

            btn_back_pred.click(
                reset_predict,
                outputs=[predict_upload, predict_results, in_audio, in_url, out_res, pred_error]
            )


        # --- TAB 4: SETTINGS ---
        with gr.Tab("⚙️ Settings"):
            gr.Markdown("### Model Management")

            # Model list view
            model_list_html = gr.HTML(label="Saved Models")

            # Model selector for actions
            gr.Markdown("### Model Actions")
            selected_model = gr.Dropdown(
                label="Select Model",
                choices=get_models_list(),
                info="Choose a model to view details or perform actions"
            )

            refresh_models_btn = gr.Button("🔄 Refresh List", size="sm")

            # Model details
            model_details = gr.Markdown(label="Model Details")

            # Action buttons
            with gr.Row():
                set_active_btn = gr.Button("✅ Set as Active Model", variant="primary")
                delete_model_btn = gr.Button("🗑️ Delete This Model", variant="stop")

            action_result = gr.Markdown(label="Result")

            gr.Markdown("---")
            gr.Markdown("### Danger Zone")
            gr.Markdown("**Warning:** This will delete ALL models and reset the system.")

            del_all_btn = gr.Button("🗑️ Delete All Models", variant="stop")
            del_all_msg = gr.Markdown(label="Status")

            def show_model_details(model_name):
                if not model_name:
                    return "Select a model to view details"

                meta = get_model_metadata(model_name)
                wins = meta.get("wins", 0)
                losses = meta.get("losses", 0)
                total = wins + losses
                win_rate = (wins / total * 100) if total > 0 else 0

                battles_text = ""
                if meta.get("battles"):
                    battles_text = "\n\n**Recent Battles:**\n"
                    for b in meta["battles"][-5:]:  # Last 5 battles
                        emoji = "🏆" if b["result"] == "win" else "❌"
                        battles_text += f"- {emoji} {b['result'].upper()} vs {b['opponent']} ({b['timestamp']})\n"

                return f"""
## {model_name}

**Validation Accuracy:** {meta.get('accuracy', 'N/A')}
**Training Epoch:** {meta.get('epoch', 'N/A')}

**Battle Statistics:**
- Total Battles: {total}
- Wins: {wins}
- Losses: {losses}
- Win Rate: {win_rate:.1f}%
{battles_text}
"""

            def refresh_all_models():
                models = get_models_list()
                html = get_model_list_html()
                return gr.update(choices=models), html

            # Load initial model list
            demo.load(
                fn=lambda: get_model_list_html(),
                outputs=model_list_html
            )

            refresh_models_btn.click(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

            selected_model.change(
                show_model_details,
                inputs=selected_model,
                outputs=model_details
            )

            set_active_btn.click(
                set_active_model,
                inputs=selected_model,
                outputs=action_result
            )

            delete_model_btn.click(
                delete_specific_model,
                inputs=selected_model,
                outputs=action_result
            ).then(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

            del_all_btn.click(delete_all_models, outputs=del_all_msg).then(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

        # --- TAB 5: CONFIGURATION ---
        with gr.Tab("📋 Configuration"):
            gr.Markdown(f"""
            ### Current System Configuration

            All settings are defined as constants in the code. To modify them, edit the source file.

            #### Model Architecture
            - **Input Shape:** MFCC {IMG_HEIGHT}x{IMG_WIDTH}
            - **Architecture:** 3 Convolutional blocks + Dense layers
            - **L2 Regularization:** {L2_REGULARIZATION}
            - **Dropout Rates:** 0.3-0.5
            - **Activation:** LeakyReLU (alpha=0.1)

            #### Training Parameters
            - **Batch Size:** {BATCH_SIZE}
            - **Epochs:** {EPOCHS}
            - **Learning Rate:** {LEARNING_RATE}
            - **Label Smoothing:** {LABEL_SMOOTHING}
            - **Early Stopping Patience:** {EARLY_STOPPING_PATIENCE} epochs
            - **LR Reduction Patience:** {LR_REDUCTION_PATIENCE} epochs
            - **LR Reduction Factor:** {LR_REDUCTION_FACTOR}

            #### Audio Processing
            - **Sample Rate:** {SAMPLE_RATE} Hz
            - **Segment Duration:** {DURATION_SECONDS} seconds
            - **MFCC Bands:** {N_MFCC}
            - **Hop Length:** {HOP_LENGTH}

            #### Data Augmentation
            - **Noise Factor:** {NOISE_FACTOR}
            - **Time Stretch Range:** {TIME_STRETCH_MIN} - {TIME_STRETCH_MAX}

            #### Prediction
            - **Multi-Segment Analysis:** {len(MULTI_SEGMENT_TIMESTAMPS)} segments
            - **Segment Positions:** {', '.join([f'{int(t*100)}%' for t in MULTI_SEGMENT_TIMESTAMPS])}
            - **Average Weight:** {SEGMENT_WEIGHT_AVG}
            - **Max Weight:** {SEGMENT_WEIGHT_MAX}

            #### File Limits
            - **Max File Size:** {MAX_FILE_SIZE_MB} MB
            """)

if __name__ == "__main__":
    if os.environ.get("BEATSENSE_AUTORUN_MAIN_TRAINING") == "1":
        print("Skipping Gradio launch during website-triggered main training.")
    else:
        demo.launch()

print("Gradio interface defined.")

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://758488585bdca544e0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio interface defined.


**Reasoning**:
The previous execution failed with a `TypeError` indicating that `demo.launch()` does not accept `theme` or `css` keyword arguments in the current Gradio environment. To resolve this error and align with the version of Gradio in use (which also produced deprecation warnings for `gr.Blocks` with these arguments), I am moving the `theme` and `css` parameters back to the `gr.Blocks()` constructor. This should resolve the `TypeError`, even if it reintroduces deprecation warnings.



In [ ]:
import gradio as gr
import os # for os.path.exists in the Gradio UI section for button visibility

# ---------------------------------------------------------
# GRADIO INTERFACE
# ---------------------------------------------------------
custom_css = """
* {
    font-family: Arial, sans-serif !important;
}

.gradio-container {
    font-family: Arial, sans-serif !important;
}
"""

with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:
    gr.Markdown("""
    # ♩&#xFE0F; BeatSense - AI Music Taste Classifier
    ### Train AI to understand your music preferences using advanced MFCC analysis
    """)

    battle_state = gr.State()
    predict_state = gr.State({"stage": "upload"})  # upload or results

    with gr.Tabs():
        # --- TAB 1: TRAIN ---
        with gr.Tab("🎓 Train Model"):
            gr.Markdown("""
            ### Training Process
            1. Loads audio files from your dataset
            2. Extracts MFCC features with proper normalization
            3. Trains a CNN model with regularization
            4. Saves a new timestamped model to model history
            """)

            with gr.Row():
                btn_train = gr.Button("🚀 Start Training", variant="primary", size="lg")
                btn_stop = gr.Button("🛑 Stop Training", variant="stop", size="lg")

            status_box = gr.Markdown(label="Training Status")
            plot_box = gr.Plot(label="Training Metrics")

            btn_train.click(train_logic, outputs=[status_box, plot_box])
            btn_stop.click(stop_training_func, outputs=status_box)

        # --- TAB 2: ARENA ---
        with gr.Tab("⚔️ Model Arena"):
            arena_stage = gr.State("setup")  # setup or results

            # Setup screen
            with gr.Group(visible=True) as arena_setup:
                gr.Markdown("### Select Models to Battle")

                with gr.Row():
                    mod_a = gr.Dropdown(
                        label="🔵 Model A",
                        choices=get_models_list(),
                        info="Select first model"
                    )
                    mod_b = gr.Dropdown(
                        label="🔴 Model B",
                        choices=get_models_list(),
                        info="Select second model"
                    )

                ref_btn = gr.Button("🔄 Refresh Model List", size="sm")

                gr.Markdown("### Upload Audio or Provide YouTube Link")
                with gr.Row():
                    bat_audio = gr.Audio(label="Audio File", type="filepath")
                    bat_url = gr.Textbox(
                        label="YouTube URL",
                        placeholder="https://www.youtube.com/watch?v=...",
                        info="Alternative to file upload"
                    )

                fight_btn = gr.Button("⚔️ START BATTLE!", variant="primary", size="lg")

            # Results screen
            with gr.Group(visible=False) as arena_results:
                bat_res = gr.Markdown(label="Battle Results")
                bat_plot = gr.Plot(label="Score Comparison")

                gr.Markdown("### What is the correct label for this song?")

                with gr.Row():
                    btn_real_like = gr.Button("👍 I LIKE This Song", variant="primary", size="lg")
                    btn_real_dis = gr.Button("👎 I DISLIKE This Song", variant="secondary", size="lg")

                judge_res = gr.Markdown(label="Winner")
                btn_back_arena = gr.Button("↩️ New Battle", size="lg")

            def show_battle_results(model_a, model_b, file, url):
                result_text, fig, battle_data = run_battle(model_a, model_b, file, url)
                return (
                    gr.update(visible=False),  # hide setup
                    gr.update(visible=True),   # show results
                    result_text,
                    fig,
                    battle_data
                )

            def reset_arena():
                return (
                    gr.update(visible=True),   # show setup
                    gr.update(visible=False),  # hide results
                    "",  # clear results
                    None,  # clear plot
                    ""   # clear judge
                )

            def resolve_and_record(label_truth, battle_data):
                result = resolve_winner(label_truth, battle_data)
                if battle_data and "Winner:" in result:
                    name_a, score_a, name_b, score_b = battle_data
                    if label_truth == "LIKE":
                        winner = name_a if score_a > score_b else name_b
                        loser = name_b if score_a > score_b else name_a
                    else:
                        winner = name_a if score_a < score_b else name_b
                        loser = name_b if score_a < score_b else name_a
                    update_battle_history(winner, loser)
                return result

            ref_btn.click(refresh_list, outputs=[mod_a, mod_b])

            fight_btn.click(
                show_battle_results,
                inputs=[mod_a, mod_b, bat_audio, bat_url],
                outputs=[arena_setup, arena_results, bat_res, bat_plot, battle_state]
            )

            btn_real_like.click(
                fn=lambda s: resolve_and_record("LIKE", s),
                inputs=battle_state,
                outputs=judge_res
            )
            btn_real_dis.click(
                fn=lambda s: resolve_and_record("DISLIKE", s),
                inputs=battle_state,
                outputs=judge_res
            )

            btn_back_arena.click(
                reset_arena,
                outputs=[arena_setup, arena_results, bat_res, bat_plot, judge_res]
            )

               # --- TAB 3: PREDICT ---
        with gr.Tab("🔎 Predict"):
            # Upload screen
            with gr.Group(visible=True) as predict_upload:
                gr.Markdown("### Analyze New Music")
                gr.Markdown("Upload an audio file or provide a YouTube link to get AI predictions.")

                with gr.Row():
                    in_audio = gr.Audio(label="Upload Audio File", type="filepath")
                    in_url = gr.Textbox(
                        label="Or Enter YouTube URL",
                        placeholder="https://www.youtube.com/watch?v=...",
                        info="Paste YouTube link"
                    )

                pred_error = gr.Markdown(value="")  # NEW: error box in upload screen
                btn_pred = gr.Button("🎯 Analyze Music", variant="primary", size="lg")

            # Results screen
            with gr.Group(visible=False) as predict_results:
                out_res = gr.Markdown(label="Analysis Results")
                btn_back_pred = gr.Button("↩️ Analyze Another Song", size="lg")

            def show_prediction_results(audio_path, url):
                # default UI state: stay on upload unless success
                # clear previous error
                if (not audio_path) and (not (url and url.strip())):
                    return (
                        gr.update(visible=True),    # keep upload visible
                        gr.update(visible=False),   # keep results hidden
                        "",                         # clear results text
                        "❌ Please provide audio file or YouTube URL"
                    )

                # If URL provided - validate and download
                target = audio_path
                if url and url.strip():
                    if not validate_url(url):
                        return (
                            gr.update(visible=True),
                            gr.update(visible=False),
                            "",
                            "❌ Invalid YouTube URL format"
                        )

                    dl_path = os.path.join(BASE_PROJECT_PATH, "temp_pred.wav")
                    success, message = download_youtube_audio(url, dl_path)
                    if not success:
                        return (
                            gr.update(visible=True),
                            gr.update(visible=False),
                            "",
                            f"❌ {message}"
                        )
                    target = dl_path

                # Validate local file
                is_valid, error = validate_audio_file(target)
                if not is_valid:
                    return (
                        gr.update(visible=True),
                        gr.update(visible=False),
                        "",
                        f"❌ {error}"
                    )

                # Run prediction
                result = predict_one(target, "")

                # Show results screen on success
                return (
                    gr.update(visible=False),  # hide upload
                    gr.update(visible=True),   # show results
                    result,                    # results text
                    ""                         # clear error
                )

            def reset_predict():
                # Go back to upload screen and clear inputs/outputs
                return (
                    gr.update(visible=True),   # show upload
                    gr.update(visible=False),  # hide results
                    None,                      # clear audio component
                    "",                        # clear url textbox
                    "",                        # clear results markdown
                    ""                         # clear error markdown
                )

            # Note: outputs now include in_audio, in_url, out_res, pred_error
            btn_pred.click(
                show_prediction_results,
                inputs=[in_audio, in_url],
                outputs=[predict_upload, predict_results, out_res, pred_error]
            )

            btn_back_pred.click(
                reset_predict,
                outputs=[predict_upload, predict_results, in_audio, in_url, out_res, pred_error]
            )


        # --- TAB 4: SETTINGS ---
        with gr.Tab("⚙️ Settings"):
            gr.Markdown("### Model Management")

            # Model list view
            model_list_html = gr.HTML(label="Saved Models")

            # Model selector for actions
            gr.Markdown("### Model Actions")
            selected_model = gr.Dropdown(
                label="Select Model",
                choices=get_models_list(),
                info="Choose a model to view details or perform actions"
            )

            refresh_models_btn = gr.Button("🔄 Refresh List", size="sm")

            # Model details
            model_details = gr.Markdown(label="Model Details")

            # Action buttons
            with gr.Row():
                set_active_btn = gr.Button("✅ Set as Active Model", variant="primary")
                delete_model_btn = gr.Button("🗑️ Delete This Model", variant="stop")

            action_result = gr.Markdown(label="Result")

            gr.Markdown("---")
            gr.Markdown("### Danger Zone")
            gr.Markdown("**Warning:** This will delete ALL models and reset the system.")

            del_all_btn = gr.Button("🗑️ Delete All Models", variant="stop")
            del_all_msg = gr.Markdown(label="Status")

            def show_model_details(model_name):
                if not model_name:
                    return "Select a model to view details"

                meta = get_model_metadata(model_name)
                wins = meta.get("wins", 0)
                losses = meta.get("losses", 0)
                total = wins + losses
                win_rate = (wins / total * 100) if total > 0 else 0

                battles_text = ""
                if meta.get("battles"):
                    battles_text = "\n\n**Recent Battles:**\n"
                    for b in meta["battles"][-5:]:  # Last 5 battles
                        emoji = "🏆" if b["result"] == "win" else "❌"
                        battles_text += f"- {emoji} {b['result'].upper()} vs {b['opponent']} ({b['timestamp']})\n"

                return f"""
## {model_name}

**Validation Accuracy:** {meta.get('accuracy', 'N/A')}
**Training Epoch:** {meta.get('epoch', 'N/A')}

**Battle Statistics:**
- Total Battles: {total}
- Wins: {wins}
- Losses: {losses}
- Win Rate: {win_rate:.1f}%
{battles_text}
"""

            def refresh_all_models():
                models = get_models_list()
                html = get_model_list_html()
                return gr.update(choices=models), html

            # Load initial model list
            demo.load(
                fn=lambda: get_model_list_html(),
                outputs=model_list_html
            )

            refresh_models_btn.click(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

            selected_model.change(
                show_model_details,
                inputs=selected_model,
                outputs=model_details
            )

            set_active_btn.click(
                set_active_model,
                inputs=selected_model,
                outputs=action_result
            )

            delete_model_btn.click(
                delete_specific_model,
                inputs=selected_model,
                outputs=action_result
            ).then(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

            del_all_btn.click(delete_all_models, outputs=del_all_msg).then(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

        # --- TAB 5: CONFIGURATION ---
        with gr.Tab("📋 Configuration"):
            gr.Markdown(f"""
            ### Current System Configuration

            All settings are defined as constants in the code. To modify them, edit the source file.

            #### Model Architecture
            - **Input Shape:** MFCC {IMG_HEIGHT}x{IMG_WIDTH}
            - **Architecture:** 3 Convolutional blocks + Dense layers
            - **L2 Regularization:** {L2_REGULARIZATION}
            - **Dropout Rates:** 0.3-0.5
            - **Activation:** LeakyReLU (alpha=0.1)

            #### Training Parameters
            - **Batch Size:** {BATCH_SIZE}
            - **Epochs:** {EPOCHS}
            - **Learning Rate:** {LEARNING_RATE}
            - **Label Smoothing:** {LABEL_SMOOTHING}
            - **Early Stopping Patience:** {EARLY_STOPPING_PATIENCE} epochs
            - **LR Reduction Patience:** {LR_REDUCTION_PATIENCE} epochs
            - **LR Reduction Factor:** {LR_REDUCTION_FACTOR}

            #### Audio Processing
            - **Sample Rate:** {SAMPLE_RATE} Hz
            - **Segment Duration:** {DURATION_SECONDS} seconds
            - **MFCC Bands:** {N_MFCC}
            - **Hop Length:** {HOP_LENGTH}

            #### Data Augmentation
            - **Noise Factor:** {NOISE_FACTOR}
            - **Time Stretch Range:** {TIME_STRETCH_MIN} - {TIME_STRETCH_MAX}

            #### Prediction
            - **Multi-Segment Analysis:** {len(MULTI_SEGMENT_TIMESTAMPS)} segments
            - **Segment Positions:** {', '.join([f'{int(t*100)}%' for t in MULTI_SEGMENT_TIMESTAMPS])}
            - **Average Weight:** {SEGMENT_WEIGHT_AVG}
            - **Max Weight:** {SEGMENT_WEIGHT_MAX}

            #### File Limits
            - **Max File Size:** {MAX_FILE_SIZE_MB} MB
            """)

if __name__ == "__main__":
    if os.environ.get("BEATSENSE_AUTORUN_MAIN_TRAINING") == "1":
        print("Skipping Gradio launch during website-triggered main training.")
    else:
        demo.launch()

print("Gradio interface defined.")

/tmp/ipykernel_1296/1098060720.py:17: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:
/tmp/ipykernel_1296/1098060720.py:17: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1b2663598847d0dfc5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio interface defined.


**Reasoning**:
The `TypeError` from the previous execution has been resolved by moving the `theme` and `css` parameters back to the `gr.Blocks()` constructor. The code now executes successfully, although deprecation warnings regarding these parameters are still present, indicating a future API change in Gradio. The task to fix the `TypeError` is complete.



In [ ]:
import gradio as gr
import os # for os.path.exists in the Gradio UI section for button visibility

# ---------------------------------------------------------
# GRADIO INTERFACE
# ---------------------------------------------------------
custom_css = """
* {
    font-family: Arial, sans-serif !important;
}

.gradio-container {
    font-family: Arial, sans-serif !important;
}
"""

with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:
    gr.Markdown("""
    # ♩&#xFE0F; BeatSense - AI Music Taste Classifier
    ### Train AI to understand your music preferences using advanced MFCC analysis
    """)

    battle_state = gr.State()
    predict_state = gr.State({"stage": "upload"})  # upload or results

    with gr.Tabs():
        # --- TAB 1: TRAIN ---
        with gr.Tab("🎓 Train Model"):
            gr.Markdown("""
            ### Training Process
            1. Loads audio files from your dataset
            2. Extracts MFCC features with proper normalization
            3. Trains a CNN model with regularization
            4. Saves a new timestamped model to model history
            """)

            with gr.Row():
                btn_train = gr.Button("🚀 Start Training", variant="primary", size="lg")
                btn_stop = gr.Button("🛑 Stop Training", variant="stop", size="lg")

            status_box = gr.Markdown(label="Training Status")
            plot_box = gr.Plot(label="Training Metrics")

            btn_train.click(train_logic, outputs=[status_box, plot_box])
            btn_stop.click(stop_training_func, outputs=status_box)

        # --- TAB 2: ARENA ---
        with gr.Tab("⚔️ Model Arena"):
            arena_stage = gr.State("setup")  # setup or results

            # Setup screen
            with gr.Group(visible=True) as arena_setup:
                gr.Markdown("### Select Models to Battle")

                with gr.Row():
                    mod_a = gr.Dropdown(
                        label="🔵 Model A",
                        choices=get_models_list(),
                        info="Select first model"
                    )
                    mod_b = gr.Dropdown(
                        label="🔴 Model B",
                        choices=get_models_list(),
                        info="Select second model"
                    )

                ref_btn = gr.Button("🔄 Refresh Model List", size="sm")

                gr.Markdown("### Upload Audio or Provide YouTube Link")
                with gr.Row():
                    bat_audio = gr.Audio(label="Audio File", type="filepath")
                    bat_url = gr.Textbox(
                        label="YouTube URL",
                        placeholder="https://www.youtube.com/watch?v=...",
                        info="Alternative to file upload"
                    )

                fight_btn = gr.Button("⚔️ START BATTLE!", variant="primary", size="lg")

            # Results screen
            with gr.Group(visible=False) as arena_results:
                bat_res = gr.Markdown(label="Battle Results")
                bat_plot = gr.Plot(label="Score Comparison")

                gr.Markdown("### What is the correct label for this song?")

                with gr.Row():
                    btn_real_like = gr.Button("👍 I LIKE This Song", variant="primary", size="lg")
                    btn_real_dis = gr.Button("👎 I DISLIKE This Song", variant="secondary", size="lg")

                judge_res = gr.Markdown(label="Winner")
                btn_back_arena = gr.Button("↩️ New Battle", size="lg")

            def show_battle_results(model_a, model_b, file, url):
                result_text, fig, battle_data = run_battle(model_a, model_b, file, url)
                return (
                    gr.update(visible=False),  # hide setup
                    gr.update(visible=True),   # show results
                    result_text,
                    fig,
                    battle_data
                )

            def reset_arena():
                return (
                    gr.update(visible=True),   # show setup
                    gr.update(visible=False),  # hide results
                    "",  # clear results
                    None,  # clear plot
                    ""   # clear judge
                )

            def resolve_and_record(label_truth, battle_data):
                result = resolve_winner(label_truth, battle_data)
                if battle_data and "Winner:" in result:
                    name_a, score_a, name_b, score_b = battle_data
                    if label_truth == "LIKE":
                        winner = name_a if score_a > score_b else name_b
                        loser = name_b if score_a > score_b else name_a
                    else:
                        winner = name_a if score_a < score_b else name_b
                        loser = name_b if score_a < score_b else name_a
                    update_battle_history(winner, loser)
                return result

            ref_btn.click(refresh_list, outputs=[mod_a, mod_b])

            fight_btn.click(
                show_battle_results,
                inputs=[mod_a, mod_b, bat_audio, bat_url],
                outputs=[arena_setup, arena_results, bat_res, bat_plot, battle_state]
            )

            btn_real_like.click(
                fn=lambda s: resolve_and_record("LIKE", s),
                inputs=battle_state,
                outputs=judge_res
            )
            btn_real_dis.click(
                fn=lambda s: resolve_and_record("DISLIKE", s),
                inputs=battle_state,
                outputs=judge_res
            )

            btn_back_arena.click(
                reset_arena,
                outputs=[arena_setup, arena_results, bat_res, bat_plot, judge_res]
            )

               # --- TAB 3: PREDICT ---
        with gr.Tab("🔎 Predict"):
            # Upload screen
            with gr.Group(visible=True) as predict_upload:
                gr.Markdown("### Analyze New Music")
                gr.Markdown("Upload an audio file or provide a YouTube link to get AI predictions.")

                with gr.Row():
                    in_audio = gr.Audio(label="Upload Audio File", type="filepath")
                    in_url = gr.Textbox(
                        label="Or Enter YouTube URL",
                        placeholder="https://www.youtube.com/watch?v=...",
                        info="Paste YouTube link"
                    )

                pred_error = gr.Markdown(value="")  # NEW: error box in upload screen
                btn_pred = gr.Button("🎯 Analyze Music", variant="primary", size="lg")

            # Results screen
            with gr.Group(visible=False) as predict_results:
                out_res = gr.Markdown(label="Analysis Results")
                btn_back_pred = gr.Button("↩️ Analyze Another Song", size="lg")

            def show_prediction_results(audio_path, url):
                # default UI state: stay on upload unless success
                # clear previous error
                if (not audio_path) and (not (url and url.strip())):
                    return (
                        gr.update(visible=True),    # keep upload visible
                        gr.update(visible=False),   # keep results hidden
                        "",                         # clear results text
                        "❌ Please provide audio file or YouTube URL"
                    )

                # If URL provided - validate and download
                target = audio_path
                if url and url.strip():
                    if not validate_url(url):
                        return (
                            gr.update(visible=True),
                            gr.update(visible=False),
                            "",
                            "❌ Invalid YouTube URL format"
                        )

                    dl_path = os.path.join(BASE_PROJECT_PATH, "temp_pred.wav")
                    success, message = download_youtube_audio(url, dl_path)
                    if not success:
                        return (
                            gr.update(visible=True),
                            gr.update(visible=False),
                            "",
                            f"❌ {message}"
                        )
                    target = dl_path

                # Validate local file
                is_valid, error = validate_audio_file(target)
                if not is_valid:
                    return (
                        gr.update(visible=True),
                        gr.update(visible=False),
                        "",
                        f"❌ {error}"
                    )

                # Run prediction
                result = predict_one(target, "")

                # Show results screen on success
                return (
                    gr.update(visible=False),  # hide upload
                    gr.update(visible=True),   # show results
                    result,                    # results text
                    ""                         # clear error
                )

            def reset_predict():
                # Go back to upload screen and clear inputs/outputs
                return (
                    gr.update(visible=True),   # show upload
                    gr.update(visible=False),  # hide results
                    None,                      # clear audio component
                    "",                        # clear url textbox
                    "",                        # clear results markdown
                    ""                         # clear error markdown
                )

            # Note: outputs now include in_audio, in_url, out_res, pred_error
            btn_pred.click(
                show_prediction_results,
                inputs=[in_audio, in_url],
                outputs=[predict_upload, predict_results, out_res, pred_error]
            )

            btn_back_pred.click(
                reset_predict,
                outputs=[predict_upload, predict_results, in_audio, in_url, out_res, pred_error]
            )


        # --- TAB 4: SETTINGS ---
        with gr.Tab("⚙️ Settings"):
            gr.Markdown("### Model Management")

            # Model list view
            model_list_html = gr.HTML(label="Saved Models")

            # Model selector for actions
            gr.Markdown("### Model Actions")
            selected_model = gr.Dropdown(
                label="Select Model",
                choices=get_models_list(),
                info="Choose a model to view details or perform actions"
            )

            refresh_models_btn = gr.Button("🔄 Refresh List", size="sm")

            # Model details
            model_details = gr.Markdown(label="Model Details")

            # Action buttons
            with gr.Row():
                set_active_btn = gr.Button("✅ Set as Active Model", variant="primary")
                delete_model_btn = gr.Button("🗑️ Delete This Model", variant="stop")

            action_result = gr.Markdown(label="Result")

            gr.Markdown("---")
            gr.Markdown("### Danger Zone")
            gr.Markdown("**Warning:** This will delete ALL models and reset the system.")

            del_all_btn = gr.Button("🗑️ Delete All Models", variant="stop")
            del_all_msg = gr.Markdown(label="Status")

            def show_model_details(model_name):
                if not model_name:
                    return "Select a model to view details"

                meta = get_model_metadata(model_name)
                wins = meta.get("wins", 0)
                losses = meta.get("losses", 0)
                total = wins + losses
                win_rate = (wins / total * 100) if total > 0 else 0

                battles_text = ""
                if meta.get("battles"):
                    battles_text = "\n\n**Recent Battles:**\n"
                    for b in meta["battles"][-5:]:  # Last 5 battles
                        emoji = "🏆" if b["result"] == "win" else "❌"
                        battles_text += f"- {emoji} {b['result'].upper()} vs {b['opponent']} ({b['timestamp']})\n"

                return f"""
## {model_name}

**Validation Accuracy:** {meta.get('accuracy', 'N/A')}
**Training Epoch:** {meta.get('epoch', 'N/A')}

**Battle Statistics:**
- Total Battles: {total}
- Wins: {wins}
- Losses: {losses}
- Win Rate: {win_rate:.1f}%
{battles_text}
"""

            def refresh_all_models():
                models = get_models_list()
                html = get_model_list_html()
                return gr.update(choices=models), html

            # Load initial model list
            demo.load(
                fn=lambda: get_model_list_html(),
                outputs=model_list_html
            )

            refresh_models_btn.click(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

            selected_model.change(
                show_model_details,
                inputs=selected_model,
                outputs=model_details
            )

            set_active_btn.click(
                set_active_model,
                inputs=selected_model,
                outputs=action_result
            )

            delete_model_btn.click(
                delete_specific_model,
                inputs=selected_model,
                outputs=action_result
            ).then(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

            del_all_btn.click(delete_all_models, outputs=del_all_msg).then(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

        # --- TAB 5: CONFIGURATION ---
        with gr.Tab("📋 Configuration"):
            gr.Markdown(f"""
            ### Current System Configuration

            All settings are defined as constants in the code. To modify them, edit the source file.

            #### Model Architecture
            - **Input Shape:** MFCC {IMG_HEIGHT}x{IMG_WIDTH}
            - **Architecture:** 3 Convolutional blocks + Dense layers
            - **L2 Regularization:** {L2_REGULARIZATION}
            - **Dropout Rates:** 0.3-0.5
            - **Activation:** LeakyReLU (alpha=0.1)

            #### Training Parameters
            - **Batch Size:** {BATCH_SIZE}
            - **Epochs:** {EPOCHS}
            - **Learning Rate:** {LEARNING_RATE}
            - **Label Smoothing:** {LABEL_SMOOTHING}
            - **Early Stopping Patience:** {EARLY_STOPPING_PATIENCE} epochs
            - **LR Reduction Patience:** {LR_REDUCTION_PATIENCE} epochs
            - **LR Reduction Factor:** {LR_REDUCTION_FACTOR}

            #### Audio Processing
            - **Sample Rate:** {SAMPLE_RATE} Hz
            - **Segment Duration:** {DURATION_SECONDS} seconds
            - **MFCC Bands:** {N_MFCC}
            - **Hop Length:** {HOP_LENGTH}

            #### Data Augmentation
            - **Noise Factor:** {NOISE_FACTOR}
            - **Time Stretch Range:** {TIME_STRETCH_MIN} - {TIME_STRETCH_MAX}

            #### Prediction
            - **Multi-Segment Analysis:** {len(MULTI_SEGMENT_TIMESTAMPS)} segments
            - **Segment Positions:** {', '.join([f'{int(t*100)}%' for t in MULTI_SEGMENT_TIMESTAMPS])}
            - **Average Weight:** {SEGMENT_WEIGHT_AVG}
            - **Max Weight:** {SEGMENT_WEIGHT_MAX}

            #### File Limits
            - **Max File Size:** {MAX_FILE_SIZE_MB} MB
            """)

if __name__ == "__main__":
    if os.environ.get("BEATSENSE_AUTORUN_MAIN_TRAINING") == "1":
        print("Skipping Gradio launch during website-triggered main training.")
    else:
        demo.launch()

print("Gradio interface defined.")

/tmp/ipykernel_1296/1098060720.py:17: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:
/tmp/ipykernel_1296/1098060720.py:17: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a85b8b5c0d661c777e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio interface defined.


In [ ]:
import gradio as gr
import os # for os.path.exists in the Gradio UI section for button visibility

# ---------------------------------------------------------
# GRADIO INTERFACE
# ---------------------------------------------------------
custom_css = """
* {
    font-family: Arial, sans-serif !important;
}

.gradio-container {
    font-family: Arial, sans-serif !important;
}
"""

with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:
    gr.Markdown("""
    # ♩&#xFE0F; BeatSense - AI Music Taste Classifier
    ### Train AI to understand your music preferences using advanced MFCC analysis
    """)

    battle_state = gr.State()
    predict_state = gr.State({"stage": "upload"})  # upload or results

    with gr.Tabs():
        # --- TAB 1: TRAIN ---
        with gr.Tab("🎓 Train Model"):
            gr.Markdown("""
            ### Training Process
            1. Loads audio files from your dataset
            2. Extracts MFCC features with proper normalization
            3. Trains a CNN model with regularization
            4. Saves a new timestamped model to model history
            """)

            with gr.Row():
                btn_train = gr.Button("🚀 Start Training", variant="primary", size="lg")
                btn_stop = gr.Button("🛑 Stop Training", variant="stop", size="lg")

            status_box = gr.Markdown(label="Training Status")
            plot_box = gr.Plot(label="Training Metrics")

            btn_train.click(train_logic, outputs=[status_box, plot_box])
            btn_stop.click(stop_training_func, outputs=status_box)

        # --- TAB 2: ARENA ---
        with gr.Tab("⚔️ Model Arena"):
            arena_stage = gr.State("setup")  # setup or results

            # Setup screen
            with gr.Group(visible=True) as arena_setup:
                gr.Markdown("### Select Models to Battle")

                with gr.Row():
                    mod_a = gr.Dropdown(
                        label="🔵 Model A",
                        choices=get_models_list(),
                        info="Select first model"
                    )
                    mod_b = gr.Dropdown(
                        label="🔴 Model B",
                        choices=get_models_list(),
                        info="Select second model"
                    )

                ref_btn = gr.Button("🔄 Refresh Model List", size="sm")

                gr.Markdown("### Upload Audio or Provide YouTube Link")
                with gr.Row():
                    bat_audio = gr.Audio(label="Audio File", type="filepath")
                    bat_url = gr.Textbox(
                        label="YouTube URL",
                        placeholder="https://www.youtube.com/watch?v=...",
                        info="Alternative to file upload"
                    )

                fight_btn = gr.Button("⚔️ START BATTLE!", variant="primary", size="lg")

            # Results screen
            with gr.Group(visible=False) as arena_results:
                bat_res = gr.Markdown(label="Battle Results")
                bat_plot = gr.Plot(label="Score Comparison")

                gr.Markdown("### What is the correct label for this song?")

                with gr.Row():
                    btn_real_like = gr.Button("👍 I LIKE This Song", variant="primary", size="lg")
                    btn_real_dis = gr.Button("👎 I DISLIKE This Song", variant="secondary", size="lg")

                judge_res = gr.Markdown(label="Winner")
                btn_back_arena = gr.Button("↩️ New Battle", size="lg")

            def show_battle_results(model_a, model_b, file, url):
                result_text, fig, battle_data = run_battle(model_a, model_b, file, url)
                return (
                    gr.update(visible=False),  # hide setup
                    gr.update(visible=True),   # show results
                    result_text,
                    fig,
                    battle_data
                )

            def reset_arena():
                return (
                    gr.update(visible=True),   # show setup
                    gr.update(visible=False),  # hide results
                    "",  # clear results
                    None,  # clear plot
                    ""   # clear judge
                )

            def resolve_and_record(label_truth, battle_data):
                result = resolve_winner(label_truth, battle_data)
                if battle_data and "Winner:" in result:
                    name_a, score_a, name_b, score_b = battle_data
                    if label_truth == "LIKE":
                        winner = name_a if score_a > score_b else name_b
                        loser = name_b if score_a > score_b else name_a
                    else:
                        winner = name_a if score_a < score_b else name_b
                        loser = name_b if score_a < score_b else name_a
                    update_battle_history(winner, loser)
                return result

            ref_btn.click(refresh_list, outputs=[mod_a, mod_b])

            fight_btn.click(
                show_battle_results,
                inputs=[mod_a, mod_b, bat_audio, bat_url],
                outputs=[arena_setup, arena_results, bat_res, bat_plot, battle_state]
            )

            btn_real_like.click(
                fn=lambda s: resolve_and_record("LIKE", s),
                inputs=battle_state,
                outputs=judge_res
            )
            btn_real_dis.click(
                fn=lambda s: resolve_and_record("DISLIKE", s),
                inputs=battle_state,
                outputs=judge_res
            )

            btn_back_arena.click(
                reset_arena,
                outputs=[arena_setup, arena_results, bat_res, bat_plot, judge_res]
            )

               # --- TAB 3: PREDICT ---
        with gr.Tab("🔎 Predict"):
            # Upload screen
            with gr.Group(visible=True) as predict_upload:
                gr.Markdown("### Analyze New Music")
                gr.Markdown("Upload an audio file or provide a YouTube link to get AI predictions.")

                with gr.Row():
                    in_audio = gr.Audio(label="Upload Audio File", type="filepath")
                    in_url = gr.Textbox(
                        label="Or Enter YouTube URL",
                        placeholder="https://www.youtube.com/watch?v=...",
                        info="Paste YouTube link"
                    )

                pred_error = gr.Markdown(value="")  # NEW: error box in upload screen
                btn_pred = gr.Button("🎯 Analyze Music", variant="primary", size="lg")

            # Results screen
            with gr.Group(visible=False) as predict_results:
                out_res = gr.Markdown(label="Analysis Results")
                btn_back_pred = gr.Button("↩️ Analyze Another Song", size="lg")

            def show_prediction_results(audio_path, url):
                # default UI state: stay on upload unless success
                # clear previous error
                if (not audio_path) and (not (url and url.strip())):
                    return (
                        gr.update(visible=True),    # keep upload visible
                        gr.update(visible=False),   # keep results hidden
                        "",                         # clear results text
                        "❌ Please provide audio file or YouTube URL"
                    )

                # If URL provided - validate and download
                target = audio_path
                if url and url.strip():
                    if not validate_url(url):
                        return (
                            gr.update(visible=True),
                            gr.update(visible=False),
                            "",
                            "❌ Invalid YouTube URL format"
                        )

                    dl_path = os.path.join(BASE_PROJECT_PATH, "temp_pred.wav")
                    success, message = download_youtube_audio(url, dl_path)
                    if not success:
                        return (
                            gr.update(visible=True),
                            gr.update(visible=False),
                            "",
                            f"❌ {message}"
                        )
                    target = dl_path

                # Validate local file
                is_valid, error = validate_audio_file(target)
                if not is_valid:
                    return (
                        gr.update(visible=True),
                        gr.update(visible=False),
                        "",
                        f"❌ {error}"
                    )

                # Run prediction
                result = predict_one(target, "")

                # Show results screen on success
                return (
                    gr.update(visible=False),  # hide upload
                    gr.update(visible=True),   # show results
                    result,                    # results text
                    ""                         # clear error
                )

            def reset_predict():
                # Go back to upload screen and clear inputs/outputs
                return (
                    gr.update(visible=True),   # show upload
                    gr.update(visible=False),  # hide results
                    None,                      # clear audio component
                    "",                        # clear url textbox
                    "",                        # clear results markdown
                    ""                         # clear error markdown
                )

            # Note: outputs now include in_audio, in_url, out_res, pred_error
            btn_pred.click(
                show_prediction_results,
                inputs=[in_audio, in_url],
                outputs=[predict_upload, predict_results, out_res, pred_error]
            )

            btn_back_pred.click(
                reset_predict,
                outputs=[predict_upload, predict_results, in_audio, in_url, out_res, pred_error]
            )


        # --- TAB 4: SETTINGS ---
        with gr.Tab("⚙️ Settings"):
            gr.Markdown("### Model Management")

            # Model list view
            model_list_html = gr.HTML(label="Saved Models")

            # Model selector for actions
            gr.Markdown("### Model Actions")
            selected_model = gr.Dropdown(
                label="Select Model",
                choices=get_models_list(),
                info="Choose a model to view details or perform actions"
            )

            refresh_models_btn = gr.Button("🔄 Refresh List", size="sm")

            # Model details
            model_details = gr.Markdown(label="Model Details")

            # Action buttons
            with gr.Row():
                set_active_btn = gr.Button("✅ Set as Active Model", variant="primary")
                delete_model_btn = gr.Button("🗑️ Delete This Model", variant="stop")

            action_result = gr.Markdown(label="Result")

            gr.Markdown("---")
            gr.Markdown("### Danger Zone")
            gr.Markdown("**Warning:** This will delete ALL models and reset the system.")

            del_all_btn = gr.Button("🗑️ Delete All Models", variant="stop")
            del_all_msg = gr.Markdown(label="Status")

            def show_model_details(model_name):
                if not model_name:
                    return "Select a model to view details"

                meta = get_model_metadata(model_name)
                wins = meta.get("wins", 0)
                losses = meta.get("losses", 0)
                total = wins + losses
                win_rate = (wins / total * 100) if total > 0 else 0

                battles_text = ""
                if meta.get("battles"):
                    battles_text = "\n\n**Recent Battles:**\n"
                    for b in meta["battles"][-5:]:  # Last 5 battles
                        emoji = "🏆" if b["result"] == "win" else "❌"
                        battles_text += f"- {emoji} {b['result'].upper()} vs {b['opponent']} ({b['timestamp']})\n"

                return f"""
## {model_name}

**Validation Accuracy:** {meta.get('accuracy', 'N/A')}
**Training Epoch:** {meta.get('epoch', 'N/A')}

**Battle Statistics:**
- Total Battles: {total}
- Wins: {wins}
- Losses: {losses}
- Win Rate: {win_rate:.1f}%
{battles_text}
"""

            def refresh_all_models():
                models = get_models_list()
                html = get_model_list_html()
                return gr.update(choices=models), html

            # Load initial model list
            demo.load(
                fn=lambda: get_model_list_html(),
                outputs=model_list_html
            )

            refresh_models_btn.click(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

            selected_model.change(
                show_model_details,
                inputs=selected_model,
                outputs=model_details
            )

            set_active_btn.click(
                set_active_model,
                inputs=selected_model,
                outputs=action_result
            )

            delete_model_btn.click(
                delete_specific_model,
                inputs=selected_model,
                outputs=action_result
            ).then(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

            del_all_btn.click(delete_all_models, outputs=del_all_msg).then(
                refresh_all_models,
                outputs=[selected_model, model_list_html]
            )

        # --- TAB 5: CONFIGURATION ---
        with gr.Tab("📋 Configuration"):
            gr.Markdown(f"""
            ### Current System Configuration

            All settings are defined as constants in the code. To modify them, edit the source file.

            #### Model Architecture
            - **Input Shape:** MFCC {IMG_HEIGHT}x{IMG_WIDTH}
            - **Architecture:** 3 Convolutional blocks + Dense layers
            - **L2 Regularization:** {L2_REGULARIZATION}
            - **Dropout Rates:** 0.3-0.5
            - **Activation:** LeakyReLU (alpha=0.1)

            #### Training Parameters
            - **Batch Size:** {BATCH_SIZE}
            - **Epochs:** {EPOCHS}
            - **Learning Rate:** {LEARNING_RATE}
            - **Label Smoothing:** {LABEL_SMOOTHING}
            - **Early Stopping Patience:** {EARLY_STOPPING_PATIENCE} epochs
            - **LR Reduction Patience:** {LR_REDUCTION_PATIENCE} epochs
            - **LR Reduction Factor:** {LR_REDUCTION_FACTOR}

            #### Audio Processing
            - **Sample Rate:** {SAMPLE_RATE} Hz
            - **Segment Duration:** {DURATION_SECONDS} seconds
            - **MFCC Bands:** {N_MFCC}
            - **Hop Length:** {HOP_LENGTH}

            #### Data Augmentation
            - **Noise Factor:** {NOISE_FACTOR}
            - **Time Stretch Range:** {TIME_STRETCH_MIN} - {TIME_STRETCH_MAX}

            #### Prediction
            - **Multi-Segment Analysis:** {len(MULTI_SEGMENT_TIMESTAMPS)} segments
            - **Segment Positions:** {', '.join([f'{{int(t*100)}}%' for t in MULTI_SEGMENT_TIMESTAMPS])}
            - **Average Weight:** {SEGMENT_WEIGHT_AVG}
            - **Max Weight:** {SEGMENT_WEIGHT_MAX}

            #### File Limits
            - **Max File Size:** {MAX_FILE_SIZE_MB} MB
            """)

if __name__ == "__main__":
    if os.environ.get("BEATSENSE_AUTORUN_MAIN_TRAINING") == "1":
        print("Skipping Gradio launch during website-triggered main training.")
    else:
        demo.launch()

print("Gradio interface defined.")

/tmp/ipykernel_1296/3866019102.py:17: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:
/tmp/ipykernel_1296/3866019102.py:17: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="BeatSense - AI Music Classifier", theme=gr.themes.Soft(), css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c5c3fa7b555d8908b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio interface defined.


In [ ]:
import os

# ---------------------------------------------------------
# WEBSITE AUTORUN HOOK
# ---------------------------------------------------------
if os.environ.get('BEATSENSE_AUTORUN_MAIN_TRAINING') == '1':
    print('Website-triggered main training autorun started.')
    status_message, _ = train_logic()
    print(status_message)
else:
    print('Notebook loaded without website autorun.')



Notebook loaded without website autorun.
